In [56]:
from pathlib import Path

BASE_DIR = Path.cwd()
print("Current folder:", BASE_DIR)

print("\nMaqola papkasi ichidagi fayllar:")
for f in BASE_DIR.iterdir():
    print("-", f.name)

Current folder: C:\Users\user\Desktop\Maqola

Maqola papkasi ichidagi fayllar:
- .ipynb_checkpoints
- 01_ADE_AI_Model_Training.ipynb
- alert_strategy_summary.csv
- alert_tier_summary.csv
- alert_tier_summary_leakage_reduced.csv
- alert_tier_summary_strict_leakage_reduced.csv
- all_medguard_results_tables.xlsx
- calibration_curve_leakage_reduced.csv
- calibration_curve_leakage_reduced.png
- catboost_ade_model.pkl
- catboost_info
- fairness_all_subgroups.csv
- fairness_all_subgroups_with_flags.csv
- fairness_by_age_group.csv
- fairness_by_country_top10.csv
- fairness_by_drug_count_category.csv
- fairness_by_patient_sex.csv
- fairness_summary_for_manuscript.csv
- fairness_summary_leakage_reduced.csv
- fda_adverse_events_2015_2026_CLEAN.csv
- final_analysis_summary_for_manuscript.csv
- final_output_check.csv
- leakage_sensitivity_analysis.csv
- local_high_risk_shap_explanation.csv
- local_high_risk_shap_explanation_leakage_reduced.csv
- model_results_table.csv
- primary_leakage_reduced_mod

In [57]:
import pandas as pd

csv_files = list(BASE_DIR.glob("*.csv"))

print("Topilgan CSV fayllar:")
for i, f in enumerate(csv_files, start=1):
    print(i, f.name)

DATA_PATH = csv_files[0]
print("\nTanlangan dataset:", DATA_PATH.name)

Topilgan CSV fayllar:
1 alert_strategy_summary.csv
2 alert_tier_summary.csv
3 alert_tier_summary_leakage_reduced.csv
4 alert_tier_summary_strict_leakage_reduced.csv
5 calibration_curve_leakage_reduced.csv
6 fairness_all_subgroups.csv
7 fairness_all_subgroups_with_flags.csv
8 fairness_by_age_group.csv
9 fairness_by_country_top10.csv
10 fairness_by_drug_count_category.csv
11 fairness_by_patient_sex.csv
12 fairness_summary_for_manuscript.csv
13 fairness_summary_leakage_reduced.csv
14 fda_adverse_events_2015_2026_CLEAN.csv
15 final_analysis_summary_for_manuscript.csv
16 final_output_check.csv
17 leakage_sensitivity_analysis.csv
18 local_high_risk_shap_explanation.csv
19 local_high_risk_shap_explanation_leakage_reduced.csv
20 model_results_table.csv
21 primary_leakage_reduced_model_results.csv
22 shap_feature_importance_leakage_reduced.csv
23 strict_leakage_sensitivity_analysis.csv
24 strict_output_check.csv
25 temporal_validation_results.csv
26 threshold_optimization_leakage_reduced.csv
27

In [58]:
df_sample = pd.read_csv(DATA_PATH, nrows=10000, low_memory=False)

print("Sample shape:", df_sample.shape)
print("\nUstunlar:")
for col in df_sample.columns:
    print("-", col)

df_sample.head()

Sample shape: (1, 9)

Ustunlar:
- Selected model
- Strategy
- Total potential alerts
- Active alerts shown
- Suppressed/non-interruptive alerts
- Alert burden reduction (%)
- Total serious ADE cases
- Serious ADE captured in active alerts
- Serious ADE sensitivity retained (%)


,Selected model,Strategy,Total potential alerts,Active alerts shown,Suppressed/non-interruptive alerts,Alert burden reduction (%),Total serious ADE cases,Serious ADE captured in active alerts,Serious ADE sensitivity retained (%)
0,XGBoost,AI-guided triage: Critical + High shown actively,40000,17555,22445,56.1125,18089,13360,73.85704


In [59]:
df = pd.read_csv(DATA_PATH, nrows=200000, low_memory=False)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1, 9)


,Selected model,Strategy,Total potential alerts,Active alerts shown,Suppressed/non-interruptive alerts,Alert burden reduction (%),Total serious ADE cases,Serious ADE captured in active alerts,Serious ADE sensitivity retained (%)
0,XGBoost,AI-guided triage: Critical + High shown actively,40000,17555,22445,56.1125,18089,13360,73.85704


In [60]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(".", "_")
)

print("Cleaned columns:")
for col in df.columns:
    print("-", col)

Cleaned columns:
- selected_model
- strategy
- total_potential_alerts
- active_alerts_shown
- suppressed/non_interruptive_alerts
- alert_burden_reduction_(%)
- total_serious_ade_cases
- serious_ade_captured_in_active_alerts
- serious_ade_sensitivity_retained_(%)


In [61]:
serious_related_cols = [
    c for c in df.columns
    if any(word in c for word in [
        "serious", "death", "hospital", "life", "threat",
        "disab", "congen", "outcome", "fatal"
    ])
]

print("Serious/outcome bilan bog‘liq ustunlar:")
for c in serious_related_cols:
    print("-", c)

Serious/outcome bilan bog‘liq ustunlar:
- total_serious_ade_cases
- serious_ade_captured_in_active_alerts
- serious_ade_sensitivity_retained_(%)


In [62]:
possible_label_cols = [
    c for c in df.columns
    if any(word in c for word in [
        "serious", "death", "hospital", "life_threat", "lifethreat",
        "disab", "congen", "fatal"
    ])
]

print("Label uchun topilgan ustunlar:")
for c in possible_label_cols:
    print("-", c)

print("\nHar bir ustundagi unique qiymatlar:")
for c in possible_label_cols[:20]:
    print("\n", c)
    print(df[c].value_counts(dropna=False).head(10))

Label uchun topilgan ustunlar:
- total_serious_ade_cases
- serious_ade_captured_in_active_alerts
- serious_ade_sensitivity_retained_(%)

Har bir ustundagi unique qiymatlar:

 total_serious_ade_cases
total_serious_ade_cases
18089    1
Name: count, dtype: int64

 serious_ade_captured_in_active_alerts
serious_ade_captured_in_active_alerts
13360    1
Name: count, dtype: int64

 serious_ade_sensitivity_retained_(%)
serious_ade_sensitivity_retained_(%)
73.85704    1
Name: count, dtype: int64


In [63]:
label_cols = possible_label_cols.copy()

for c in label_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["serious_ade"] = (df[label_cols].sum(axis=1) > 0).astype(int)

print("Target distribution:")
print(df["serious_ade"].value_counts())
print(df["serious_ade"].value_counts(normalize=True))

Target distribution:
serious_ade
1    1
Name: count, dtype: int64
serious_ade
1    1.0
Name: proportion, dtype: float64


In [64]:
candidate_keywords = [
    "age", "sex", "gender", "weight",
    "country", "drug", "medicinal", "route",
    "reaction", "manufacturer", "report", "year",
    "indication", "outcome"
]

candidate_features = [
    c for c in df.columns
    if any(k in c for k in candidate_keywords)
]

# label/outcome ustunlarini featurelardan olib tashlaymiz
remove_from_features = set(label_cols + ["serious_ade"])

feature_cols = [
    c for c in candidate_features
    if c not in remove_from_features
]

print("Feature ustunlar soni:", len(feature_cols))
for c in feature_cols:
    print("-", c)

Feature ustunlar soni: 0


In [65]:
model_df = df[feature_cols + ["serious_ade"]].copy()

# To‘liq bo‘sh ustunlarni olib tashlaymiz
empty_cols = [c for c in model_df.columns if model_df[c].isna().mean() > 0.98]
print("98% dan ko‘p bo‘sh ustunlar:", empty_cols)

model_df = model_df.drop(columns=empty_cols)

feature_cols = [c for c in model_df.columns if c != "serious_ade"]

print("Final model_df shape:", model_df.shape)
print("Final feature count:", len(feature_cols))
model_df.head()

98% dan ko‘p bo‘sh ustunlar: []
Final model_df shape: (1, 1)
Final feature count: 0


,serious_ade
0,1


In [66]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"C:\Users\user\Desktop\Maqola")

print("Folder exists:", BASE_DIR.exists())
print("Folder:", BASE_DIR)

print("\nPapka ichidagi fayllar:")
for f in BASE_DIR.iterdir():
    print("-", f.name, "| suffix:", f.suffix, "| size MB:", round(f.stat().st_size / 1024 / 1024, 2))

Folder exists: True
Folder: C:\Users\user\Desktop\Maqola

Papka ichidagi fayllar:
- .ipynb_checkpoints | suffix:  | size MB: 0.0
- 01_ADE_AI_Model_Training.ipynb | suffix: .ipynb | size MB: 0.24
- alert_strategy_summary.csv | suffix: .csv | size MB: 0.0
- alert_tier_summary.csv | suffix: .csv | size MB: 0.0
- alert_tier_summary_leakage_reduced.csv | suffix: .csv | size MB: 0.0
- alert_tier_summary_strict_leakage_reduced.csv | suffix: .csv | size MB: 0.0
- all_medguard_results_tables.xlsx | suffix: .xlsx | size MB: 0.03
- calibration_curve_leakage_reduced.csv | suffix: .csv | size MB: 0.0
- calibration_curve_leakage_reduced.png | suffix: .png | size MB: 0.14
- catboost_ade_model.pkl | suffix: .pkl | size MB: 10.6
- catboost_info | suffix:  | size MB: 0.0
- fairness_all_subgroups.csv | suffix: .csv | size MB: 0.01
- fairness_all_subgroups_with_flags.csv | suffix: .csv | size MB: 0.01
- fairness_by_age_group.csv | suffix: .csv | size MB: 0.0
- fairness_by_country_top10.csv | suffix: .csv 

In [67]:
possible_files = list(BASE_DIR.glob("fda_adverse_events_2015_2026_CLEAN*"))

print("Topilgan dataset fayllar:")
for f in possible_files:
    print("-", f.name, "| suffix:", f.suffix)

if len(possible_files) == 0:
    raise FileNotFoundError("Desktop\\Maqola ichida fda_adverse_events_2015_2026_CLEAN nomli dataset topilmadi.")

DATA_PATH = possible_files[0]
print("\nUsing dataset:", DATA_PATH)

if DATA_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(DATA_PATH, low_memory=False)
elif DATA_PATH.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH, low_memory=False)

print("Original df shape:", df.shape)
display(df.head())

Topilgan dataset fayllar:
- fda_adverse_events_2015_2026_CLEAN.csv | suffix: .csv

Using dataset: C:\Users\user\Desktop\Maqola\fda_adverse_events_2015_2026_CLEAN.csv
Original df shape: (528000, 30)


,report_id,receive_date,year,month,quarter,serious,serious_flags,is_fatal,is_hospitalized,is_life_threat,...,manufacturer,pharm_class,num_drugs,drug_count_category,patient_age_years,age_group,patient_sex,patient_weight_kg,country,report_age_days
0,10004718,2015-02-11,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Glaxosmithkline Llc,Unknown,7,Polypharmacy(6+),64.0,Middle-Aged(41-65),Female,NaN,US,4063
1,10004926,2015-02-13,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Bayer Healthcare Pharmaceuticals Inc.,Progestin [EPC]; Progestin-containing Intraute...,1,Single,20.0,Adult(19-40),Female,54.0,US,4061
2,10005223,2015-02-19,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Cordavis Limited; Abbvie Inc.,Unknown,14,Polypharmacy(6+),60.0,Middle-Aged(41-65),Female,NaN,US,4055
3,10005378,2015-02-17,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Unknown,Unknown,34,Polypharmacy(6+),20.0,Adult(19-40),Female,NaN,BR,4057
4,10005980,2015-02-21,2015,2,2015Q1,Yes,NaN,False,False,False,...,Live Betr Llc; American Sales Company; Little ...,Unknown,7,Polypharmacy(6+),NaN,Unknown,Female,NaN,GB,4053


In [68]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(".", "_")
)

print("df shape:", df.shape)
print("\nColumns:")
for c in df.columns:
    print("-", c)

df shape: (528000, 30)

Columns:
- report_id
- receive_date
- year
- month
- quarter
- serious
- serious_flags
- is_fatal
- is_hospitalized
- is_life_threat
- is_disabling
- reactions
- primary_reaction
- reaction_outcomes
- patient_recovered
- num_reactions
- suspect_drug
- brand_name
- drug_route
- drug_indication
- manufacturer
- pharm_class
- num_drugs
- drug_count_category
- patient_age_years
- age_group
- patient_sex
- patient_weight_kg
- country
- report_age_days


In [69]:
print("df shape:", df.shape)

print("\nBarcha ustunlar:")
for i, col in enumerate(df.columns, start=1):
    print(i, col)

df shape: (528000, 30)

Barcha ustunlar:
1 report_id
2 receive_date
3 year
4 month
5 quarter
6 serious
7 serious_flags
8 is_fatal
9 is_hospitalized
10 is_life_threat
11 is_disabling
12 reactions
13 primary_reaction
14 reaction_outcomes
15 patient_recovered
16 num_reactions
17 suspect_drug
18 brand_name
19 drug_route
20 drug_indication
21 manufacturer
22 pharm_class
23 num_drugs
24 drug_count_category
25 patient_age_years
26 age_group
27 patient_sex
28 patient_weight_kg
29 country
30 report_age_days


In [70]:
keywords = ["serious", "ade", "fatal", "death", "hospital", "life", "label", "outcome"]

matching_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in keywords)
]

print("Serious/ADE/labelga o‘xshash ustunlar:")
for c in matching_cols:
    print("-", c)

print("\nUlarning sample qiymatlari:")
for c in matching_cols:
    print("\n", "="*80)
    print(c)
    print(df[c].value_counts(dropna=False).head(20))

Serious/ADE/labelga o‘xshash ustunlar:
- serious
- serious_flags
- is_fatal
- is_hospitalized
- is_life_threat
- reaction_outcomes

Ularning sample qiymatlari:

serious
serious
Yes    395000
No     133000
Name: count, dtype: int64

serious_flags
serious_flags
NaN                                                                         301106
Hospitalization                                                             142536
Death                                                                        24407
Death; Hospitalization                                                       22828
Hospitalization; Life-Threatening                                            11163
Disability                                                                    5763
Hospitalization; Disability                                                   4544
Life-Threatening                                                              4309
Death; Hospitalization; Life-Threatening                                    

In [71]:
if "serious" in df.columns:
    df["serious_ade"] = pd.to_numeric(df["serious"], errors="coerce").fillna(0).astype(int)
    print("serious_ade serious ustunidan yaratildi.")
    print(df["serious_ade"].value_counts())
    print(df["serious_ade"].value_counts(normalize=True))
else:
    print("serious ustuni topilmadi.")

serious_ade serious ustunidan yaratildi.
serious_ade
0    528000
Name: count, dtype: int64
serious_ade
0    1.0
Name: proportion, dtype: float64


In [72]:
if "label" in df.columns:
    df["serious_ade"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
    print("serious_ade label ustunidan yaratildi.")
    print(df["serious_ade"].value_counts())
    print(df["serious_ade"].value_counts(normalize=True))
else:
    print("label ustuni topilmadi.")

label ustuni topilmadi.


In [73]:
target_col = "serious_ade"

print("serious_ade exists:", target_col in df.columns)

if target_col not in df.columns:
    raise ValueError("Hali ham serious_ade yaratilmagan. 2-cell natijasini menga yuboring.")

df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

print("df after target cleaning:", df.shape)
print(df[target_col].value_counts())
print(df[target_col].value_counts(normalize=True))

serious_ade exists: True
df after target cleaning: (528000, 31)
serious_ade
0    528000
Name: count, dtype: int64
serious_ade
0    1.0
Name: proportion, dtype: float64


In [74]:
model_df = df.copy()

feature_cols = [
    "year",
    "num_reactions",
    "num_drugs",
    "patient_age_years",
    "patient_weight_kg",
    "report_age_days",
    "reactions",
    "primary_reaction",
    "reaction_outcomes",
    "suspect_drug",
    "drug_route",
    "drug_indication",
    "manufacturer",
    "drug_count_category",
    "age_group",
    "patient_sex",
    "country"
]

feature_cols = [c for c in feature_cols if c in model_df.columns]

print("model_df shape:", model_df.shape)
print("feature_cols count:", len(feature_cols))
print(feature_cols)

model_df shape: (528000, 31)
feature_cols count: 17
['year', 'num_reactions', 'num_drugs', 'patient_age_years', 'patient_weight_kg', 'report_age_days', 'reactions', 'primary_reaction', 'reaction_outcomes', 'suspect_drug', 'drug_route', 'drug_indication', 'manufacturer', 'drug_count_category', 'age_group', 'patient_sex', 'country']


In [75]:
from sklearn.model_selection import train_test_split

X = model_df[feature_cols].copy()
y = model_df["serious_ade"].astype(int).copy()

print("X:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

X: (528000, 17)
y distribution:
serious_ade
0    528000
Name: count, dtype: int64
serious_ade
0    1.0
Name: proportion, dtype: float64
Train: (422400, 17)
Test: (105600, 17)


In [76]:
leakage_features = ["reaction_outcomes"]

feature_cols_lr = [
    c for c in feature_cols
    if c not in leakage_features
]

X_lr = model_df[feature_cols_lr].copy()
y_lr = model_df["serious_ade"].astype(int).copy()

print("Original feature count:", len(feature_cols))
print("Leakage-reduced feature count:", len(feature_cols_lr))
print("Removed:", leakage_features)
print("X_lr shape:", X_lr.shape)
print("y_lr shape:", y_lr.shape)

print(y_lr.value_counts(normalize=True))

Original feature count: 17
Leakage-reduced feature count: 16
Removed: ['reaction_outcomes']
X_lr shape: (528000, 16)
y_lr shape: (528000,)
serious_ade
0    1.0
Name: proportion, dtype: float64


In [77]:
from sklearn.model_selection import train_test_split

X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr,
    y_lr,
    test_size=0.2,
    random_state=42,
    stratify=y_lr
)

print("Train:", X_lr_train.shape)
print("Test:", X_lr_test.shape)

Train: (422400, 16)
Test: (105600, 16)


In [78]:
model_df = df.copy()

feature_cols = [
    "year",
    "num_reactions",
    "num_drugs",
    "patient_age_years",
    "patient_weight_kg",
    "report_age_days",
    "reactions",
    "primary_reaction",
    "reaction_outcomes",
    "suspect_drug",
    "drug_route",
    "drug_indication",
    "manufacturer",
    "drug_count_category",
    "age_group",
    "patient_sex",
    "country"
]

feature_cols = [c for c in feature_cols if c in model_df.columns]

print("model_df shape:", model_df.shape)
print("feature_cols count:", len(feature_cols))
print("\nFeature columns:")
for c in feature_cols:
    print("-", c)

if model_df.shape[0] < 1000:
    raise ValueError("model_df juda kichik. Haqiqiy dataset o‘qilmagan bo‘lishi mumkin.")

if len(feature_cols) == 0:
    raise ValueError("feature_cols bo‘sh. Ustun nomlari dataset bilan mos emas.")

model_df shape: (528000, 31)
feature_cols count: 17

Feature columns:
- year
- num_reactions
- num_drugs
- patient_age_years
- patient_weight_kg
- report_age_days
- reactions
- primary_reaction
- reaction_outcomes
- suspect_drug
- drug_route
- drug_indication
- manufacturer
- drug_count_category
- age_group
- patient_sex
- country


In [79]:
from sklearn.model_selection import train_test_split

X = model_df[feature_cols]
y = model_df["serious_ade"].astype(int)

print("X:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

X: (528000, 17)
y distribution:
serious_ade
0    1.0
Name: proportion, dtype: float64
Train: (422400, 17)
Test: (105600, 17)


In [80]:
if "report_id" in feature_cols:
    feature_cols.remove("report_id")

X = model_df[feature_cols]
y = model_df["serious_ade"].astype(int)

print("Final feature columns:")
for c in feature_cols:
    print("-", c)

print("X shape:", X.shape)

Final feature columns:
- year
- num_reactions
- num_drugs
- patient_age_years
- patient_weight_kg
- report_age_days
- reactions
- primary_reaction
- reaction_outcomes
- suspect_drug
- drug_route
- drug_indication
- manufacturer
- drug_count_category
- age_group
- patient_sex
- country
X shape: (528000, 17)


In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (422400, 17)
Test: (105600, 17)


In [82]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("Categorical cols:", categorical_cols)
print("Numeric cols:", numeric_cols)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

Categorical cols: ['reactions', 'primary_reaction', 'reaction_outcomes', 'suspect_drug', 'drug_route', 'drug_indication', 'manufacturer', 'drug_count_category', 'age_group', 'patient_sex', 'country']
Numeric cols: ['year', 'num_reactions', 'num_drugs', 'patient_age_years', 'patient_weight_kg', 'report_age_days']


In [83]:
print("model_df shape:", model_df.shape)
print("serious_ade exists:", "serious_ade" in model_df.columns)

print("\nserious_ade value counts:")
print(model_df["serious_ade"].value_counts(dropna=False))

print("\nserious_ade normalized:")
print(model_df["serious_ade"].value_counts(normalize=True, dropna=False))

model_df shape: (528000, 31)
serious_ade exists: True

serious_ade value counts:
serious_ade
0    528000
Name: count, dtype: int64

serious_ade normalized:
serious_ade
0    1.0
Name: proportion, dtype: float64


In [84]:
keywords = ["serious", "fatal", "death", "hospital", "life", "disab", "label", "outcome"]

matching_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in keywords)
]

print("Serious/labelga o‘xshash ustunlar:")
for c in matching_cols:
    print("-", c)

for c in matching_cols:
    print("\n" + "="*80)
    print(c)
    print(df[c].value_counts(dropna=False).head(20))

Serious/labelga o‘xshash ustunlar:
- serious
- serious_flags
- is_fatal
- is_hospitalized
- is_life_threat
- is_disabling
- reaction_outcomes
- serious_ade

serious
serious
Yes    395000
No     133000
Name: count, dtype: int64

serious_flags
serious_flags
NaN                                                                         301106
Hospitalization                                                             142536
Death                                                                        24407
Death; Hospitalization                                                       22828
Hospitalization; Life-Threatening                                            11163
Disability                                                                    5763
Hospitalization; Disability                                                   4544
Life-Threatening                                                              4309
Death; Hospitalization; Life-Threatening                                      37

In [85]:
if "serious" in df.columns:
    df["serious_ade"] = pd.to_numeric(df["serious"], errors="coerce")
    df = df.dropna(subset=["serious_ade"]).copy()
    df["serious_ade"] = df["serious_ade"].astype(int)

    print("serious_ade serious ustunidan yaratildi.")
    print(df["serious_ade"].value_counts())
    print(df["serious_ade"].value_counts(normalize=True))
else:
    print("serious ustuni topilmadi.")

serious_ade serious ustunidan yaratildi.
Series([], Name: count, dtype: int64)
Series([], Name: proportion, dtype: float64)


In [86]:
possible_flags = [
    "is_fatal",
    "is_hospitalized",
    "is_life_threat",
    "is_disabling",
    "is_congenital_anomaly",
    "required_intervention",
    "serious_other"
]

flag_cols = [c for c in possible_flags if c in df.columns]

print("Topilgan flag ustunlar:", flag_cols)

if len(flag_cols) == 0:
    raise ValueError("Serious ADE yaratish uchun flag ustunlar topilmadi.")

for c in flag_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df["serious_ade"] = (df[flag_cols].sum(axis=1) > 0).astype(int)

print("serious_ade flaglardan yaratildi.")
print(df["serious_ade"].value_counts())
print(df["serious_ade"].value_counts(normalize=True))

Topilgan flag ustunlar: ['is_fatal', 'is_hospitalized', 'is_life_threat', 'is_disabling']
serious_ade flaglardan yaratildi.
Series([], Name: count, dtype: int64)
Series([], Name: proportion, dtype: float64)


In [87]:
print("df shape:", df.shape)

keywords = [
    "serious", "fatal", "death", "hospital", "life",
    "disab", "congenital", "intervention", "label",
    "outcome", "reaction"
]

matching_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in keywords)
]

print("\nSerious/ADE/label/outcome ga o'xshash ustunlar:")
for c in matching_cols:
    print("-", c)

print("\nHar bir mos ustunning qiymatlari:")
for c in matching_cols:
    print("\n" + "="*100)
    print(c)
    print("dtype:", df[c].dtype)
    print(df[c].value_counts(dropna=False).head(30))

df shape: (0, 31)

Serious/ADE/label/outcome ga o'xshash ustunlar:
- serious
- serious_flags
- is_fatal
- is_hospitalized
- is_life_threat
- is_disabling
- reactions
- primary_reaction
- reaction_outcomes
- num_reactions
- serious_ade

Har bir mos ustunning qiymatlari:

serious
dtype: str
Series([], Name: count, dtype: int64)

serious_flags
dtype: str
Series([], Name: count, dtype: int64)

is_fatal
dtype: int64
Series([], Name: count, dtype: int64)

is_hospitalized
dtype: int64
Series([], Name: count, dtype: int64)

is_life_threat
dtype: int64
Series([], Name: count, dtype: int64)

is_disabling
dtype: int64
Series([], Name: count, dtype: int64)

reactions
dtype: str
Series([], Name: count, dtype: int64)

primary_reaction
dtype: str
Series([], Name: count, dtype: int64)

reaction_outcomes
dtype: str
Series([], Name: count, dtype: int64)

num_reactions
dtype: int64
Series([], Name: count, dtype: int64)

serious_ade
dtype: int64
Series([], Name: count, dtype: int64)


In [88]:
print("DATA_PATH:", DATA_PATH)
print("df shape:", df.shape)

print("\nBirinchi 5 qator:")
display(df.head())

print("\nUstunlar soni:", len(df.columns))
print("Ustunlar:")
for i, c in enumerate(df.columns, 1):
    print(i, c)

DATA_PATH: C:\Users\user\Desktop\Maqola\fda_adverse_events_2015_2026_CLEAN.csv
df shape: (0, 31)

Birinchi 5 qator:


,report_id,receive_date,year,month,quarter,serious,serious_flags,is_fatal,is_hospitalized,is_life_threat,...,pharm_class,num_drugs,drug_count_category,patient_age_years,age_group,patient_sex,patient_weight_kg,country,report_age_days,serious_ade



Ustunlar soni: 31
Ustunlar:
1 report_id
2 receive_date
3 year
4 month
5 quarter
6 serious
7 serious_flags
8 is_fatal
9 is_hospitalized
10 is_life_threat
11 is_disabling
12 reactions
13 primary_reaction
14 reaction_outcomes
15 patient_recovered
16 num_reactions
17 suspect_drug
18 brand_name
19 drug_route
20 drug_indication
21 manufacturer
22 pharm_class
23 num_drugs
24 drug_count_category
25 patient_age_years
26 age_group
27 patient_sex
28 patient_weight_kg
29 country
30 report_age_days
31 serious_ade


In [89]:
from pathlib import Path

DATA_PATH = Path(r"C:\Users\user\Desktop\Maqola\fda_adverse_events_2015_2026_CLEAN.csv")

print("File exists:", DATA_PATH.exists())
print("File size MB:", round(DATA_PATH.stat().st_size / 1024 / 1024, 4))

with open(DATA_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(10):
        line = f.readline()
        print(f"{i+1}:", line[:300])

File exists: True
File size MB: 177.6262
1: report_id,receive_date,year,month,quarter,serious,serious_flags,is_fatal,is_hospitalized,is_life_threat,is_disabling,reactions,primary_reaction,reaction_outcomes,patient_recovered,num_reactions,suspect_drug,brand_name,drug_route,drug_indication,manufacturer,pharm_class,num_drugs,drug_count_category,
2: 10004718,2015-02-11,2015,2,2015Q1,Yes,Hospitalization,False,True,False,False,Breast pain; Asthma; Cough; Drug ineffective,Breast pain,Unknown; Not Recovered; Recovered; Recovering,False,4,FLUTICASONE PROPIONATE AND SALMETEROL XINAFOATE,ADVAIR HFA,Unknown,Unknown,Glaxosmithkline Llc,Unknown,7,Polypha
3: 10004926,2015-02-13,2015,2,2015Q1,Yes,Hospitalization,False,True,False,False,Device issue; Scar; Uterine perforation; Abortion spontaneous; Pain,Device issue,Unknown,False,10,LEVONORGESTREL,MIRENA,15.0,Contraception,Bayer Healthcare Pharmaceuticals Inc.,Progestin [EPC]; Progestin-containing Intraute
4: 10005223,2015-02-19,2015,2,2015Q1,Yes,Hospit

In [90]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(r"C:\Users\user\Desktop\Maqola\fda_adverse_events_2015_2026_CLEAN.csv")

df = pd.read_csv(
    DATA_PATH,
    engine="python",
    on_bad_lines="skip"
)

print("df shape:", df.shape)
display(df.head())

print("\nColumns:")
for c in df.columns:
    print("-", c)

df shape: (528000, 30)


,report_id,receive_date,year,month,quarter,serious,serious_flags,is_fatal,is_hospitalized,is_life_threat,...,manufacturer,pharm_class,num_drugs,drug_count_category,patient_age_years,age_group,patient_sex,patient_weight_kg,country,report_age_days
0,10004718,2015-02-11,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Glaxosmithkline Llc,Unknown,7,Polypharmacy(6+),64.0,Middle-Aged(41-65),Female,NaN,US,4063
1,10004926,2015-02-13,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Bayer Healthcare Pharmaceuticals Inc.,Progestin [EPC]; Progestin-containing Intraute...,1,Single,20.0,Adult(19-40),Female,54.0,US,4061
2,10005223,2015-02-19,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Cordavis Limited; Abbvie Inc.,Unknown,14,Polypharmacy(6+),60.0,Middle-Aged(41-65),Female,NaN,US,4055
3,10005378,2015-02-17,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Unknown,Unknown,34,Polypharmacy(6+),20.0,Adult(19-40),Female,NaN,BR,4057
4,10005980,2015-02-21,2015,2,2015Q1,Yes,NaN,False,False,False,...,Live Betr Llc; American Sales Company; Little ...,Unknown,7,Polypharmacy(6+),NaN,Unknown,Female,NaN,GB,4053



Columns:
- report_id
- receive_date
- year
- month
- quarter
- serious
- serious_flags
- is_fatal
- is_hospitalized
- is_life_threat
- is_disabling
- reactions
- primary_reaction
- reaction_outcomes
- patient_recovered
- num_reactions
- suspect_drug
- brand_name
- drug_route
- drug_indication
- manufacturer
- pharm_class
- num_drugs
- drug_count_category
- patient_age_years
- age_group
- patient_sex
- patient_weight_kg
- country
- report_age_days


In [91]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(".", "_", regex=False)
)

print("df shape:", df.shape)
print(df.columns.tolist())

df shape: (528000, 30)
['report_id', 'receive_date', 'year', 'month', 'quarter', 'serious', 'serious_flags', 'is_fatal', 'is_hospitalized', 'is_life_threat', 'is_disabling', 'reactions', 'primary_reaction', 'reaction_outcomes', 'patient_recovered', 'num_reactions', 'suspect_drug', 'brand_name', 'drug_route', 'drug_indication', 'manufacturer', 'pharm_class', 'num_drugs', 'drug_count_category', 'patient_age_years', 'age_group', 'patient_sex', 'patient_weight_kg', 'country', 'report_age_days']


In [92]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_scaled, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

log_model = Pipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1
    ))
])

log_model.fit(X_train, y_train)

log_proba = log_model.predict_proba(X_test)[:, 1]
log_pred = (log_proba >= 0.5).astype(int)

print("Logistic Regression AUROC:", roc_auc_score(y_test, log_proba))
print("Logistic Regression AUPRC:", average_precision_score(y_test, log_proba))
print(classification_report(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))

C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

In [93]:
from xgboost import XGBClassifier

xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pred = (xgb_proba >= 0.5).astype(int)

print("XGBoost AUROC:", roc_auc_score(y_test, xgb_proba))
print("XGBoost AUPRC:", average_precision_score(y_test, xgb_proba))
print(classification_report(y_test, xgb_pred))
print(confusion_matrix(y_test, xgb_pred))

XGBoost AUROC: nan
XGBoost AUPRC: 0.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    105600

    accuracy                           1.00    105600
   macro avg       1.00      1.00      1.00    105600
weighted avg       1.00      1.00      1.00    105600

[[105600]]


C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
C:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
from catboost import CatBoostClassifier

cat_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=50,
        random_seed=42
    ))
])

cat_model.fit(X_train, y_train)

cat_proba = cat_model.predict_proba(X_test)[:, 1]
cat_pred = (cat_proba >= 0.5).astype(int)

print("CatBoost AUROC:", roc_auc_score(y_test, cat_proba))
print("CatBoost AUPRC:", average_precision_score(y_test, cat_proba))
print(classification_report(y_test, cat_pred))
print(confusion_matrix(y_test, cat_pred))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, brier_score_loss

def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else 0

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

results = []
results.append(get_metrics("Logistic Regression", y_test, log_proba))

try:
    results.append(get_metrics("XGBoost", y_test, xgb_proba))
except NameError:
    print("XGBoost hali ishlamagan")

try:
    results.append(get_metrics("CatBoost", y_test, cat_proba))
except NameError:
    print("CatBoost hali ishlamagan")

results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.to_csv("model_results_table.csv", index=False)
print("Saved: model_results_table.csv")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)
import pandas as pd
import numpy as np

def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

results = []

results.append(get_metrics("Logistic Regression", y_test, log_proba))

try:
    results.append(get_metrics("XGBoost", y_test, xgb_proba))
except NameError:
    print("XGBoost hali ishlamagan")

try:
    results.append(get_metrics("CatBoost", y_test, cat_proba))
except NameError:
    print("CatBoost hali ishlamagan")

results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.to_csv("model_results_table.csv", index=False)
print("Saved: model_results_table.csv")

In [ ]:
import joblib

joblib.dump(xgb_model, "xgboost_ade_model.pkl")
print("Saved: xgboost_ade_model.pkl")

In [ ]:
joblib.dump(cat_model, "catboost_ade_model.pkl")
print("Saved: catboost_ade_model.pkl")

In [ ]:
alert_df = X_test.copy()
alert_df["true_serious_ade"] = y_test.values
alert_df["predicted_risk"] = xgb_proba

def assign_alert_tier(p):
    if p >= 0.80:
        return "Critical"
    elif p >= 0.50:
        return "High"
    elif p >= 0.20:
        return "Advisory"
    else:
        return "Suppressed low-risk"

alert_df["alert_tier"] = alert_df["predicted_risk"].apply(assign_alert_tier)

alert_df[["predicted_risk", "alert_tier", "true_serious_ade"]].head()

In [ ]:
alert_summary = alert_df.groupby("alert_tier").agg(
    total_alerts=("alert_tier", "count"),
    serious_ade_cases=("true_serious_ade", "sum"),
    mean_predicted_risk=("predicted_risk", "mean")
).reset_index()

alert_summary["percent_of_alerts"] = alert_summary["total_alerts"] / len(alert_df) * 100
alert_summary["serious_ade_rate"] = alert_summary["serious_ade_cases"] / alert_summary["total_alerts"] * 100

alert_summary

In [ ]:
alert_summary.to_csv("alert_tier_summary.csv", index=False)
print("Saved: alert_tier_summary.csv")

In [ ]:
total_alerts = len(alert_df)

active_alerts = alert_df[alert_df["alert_tier"].isin(["Critical", "High"])].shape[0]
suppressed_or_noninterruptive = total_alerts - active_alerts

alert_burden_reduction = suppressed_or_noninterruptive / total_alerts * 100

serious_total = alert_df["true_serious_ade"].sum()
serious_captured_active = alert_df[
    (alert_df["alert_tier"].isin(["Critical", "High"])) &
    (alert_df["true_serious_ade"] == 1)
].shape[0]

serious_sensitivity_retained = serious_captured_active / serious_total * 100

print("Total potential alerts:", total_alerts)
print("Active alerts shown:", active_alerts)
print("Suppressed/non-interruptive alerts:", suppressed_or_noninterruptive)
print("Alert burden reduction (%):", alert_burden_reduction)
print("Serious ADE sensitivity retained (%):", serious_sensitivity_retained)

In [ ]:
alert_strategy_df = pd.DataFrame([{
    "Strategy": "AI-guided triage: Critical + High shown actively",
    "Total potential alerts": total_alerts,
    "Active alerts shown": active_alerts,
    "Suppressed/non-interruptive alerts": suppressed_or_noninterruptive,
    "Alert burden reduction (%)": alert_burden_reduction,
    "Serious ADE sensitivity retained (%)": serious_sensitivity_retained
}])

alert_strategy_df

In [ ]:
alert_strategy_df.to_csv("alert_strategy_summary.csv", index=False)
print("Saved: alert_strategy_summary.csv")

In [ ]:
fairness_df = X_test.copy()
fairness_df["true_serious_ade"] = y_test.values
fairness_df["predicted_risk"] = xgb_proba
fairness_df["predicted_label"] = (xgb_proba >= 0.5).astype(int)

def subgroup_metrics(data, subgroup_col):
    rows = []
    for group, g in data.groupby(subgroup_col):
        if g["true_serious_ade"].nunique() < 2:
            continue
        
        y_true_g = g["true_serious_ade"]
        y_prob_g = g["predicted_risk"]
        y_pred_g = g["predicted_label"]
        
        tn, fp, fn, tp = confusion_matrix(y_true_g, y_pred_g).ravel()
        
        rows.append({
            "Subgroup variable": subgroup_col,
            "Subgroup": group,
            "n": len(g),
            "AUROC": roc_auc_score(y_true_g, y_prob_g),
            "AUPRC": average_precision_score(y_true_g, y_prob_g),
            "Sensitivity": recall_score(y_true_g, y_pred_g, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
            "FNR": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
            "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan
        })
    return pd.DataFrame(rows)

fairness_sex = subgroup_metrics(fairness_df, "patient_sex")
fairness_sex

In [ ]:
fairness_sex.to_csv("fairness_by_patient_sex.csv", index=False)
print("Saved: fairness_by_patient_sex.csv")

In [ ]:
fairness_age = subgroup_metrics(fairness_df, "age_group")
fairness_age

In [ ]:
fairness_age.to_csv("fairness_by_age_group.csv", index=False)
print("Saved: fairness_by_age_group.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)

# =========================
# 1) results_df jadvali
# =========================

def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

results = []

# Logistic Regression
if "log_proba" in globals():
    results.append(get_metrics("Logistic Regression", y_test, log_proba))
else:
    print("log_proba topilmadi: Logistic Regression natijasi qo‘shilmadi.")

# XGBoost
if "xgb_proba" in globals():
    results.append(get_metrics("XGBoost", y_test, xgb_proba))
else:
    print("xgb_proba topilmadi: XGBoost natijasi qo‘shilmadi.")

# CatBoost
if "cat_proba" in globals():
    results.append(get_metrics("CatBoost", y_test, cat_proba))
else:
    print("cat_proba topilmadi: CatBoost natijasi qo‘shilmadi.")

results_df = pd.DataFrame(results)

print("=== results_df jadvali ===")
display(results_df)

results_df.to_csv("model_results_table.csv", index=False)
print("Saved: model_results_table.csv")


# =========================
# 2) alert_summary jadvali
# =========================

# Alert simulation uchun qaysi model probability ishlatiladi?
# Avval XGBoost, bo‘lmasa CatBoost, bo‘lmasa Logistic Regression olinadi.
if "xgb_proba" in globals():
    selected_proba = xgb_proba
    selected_model_name = "XGBoost"
elif "cat_proba" in globals():
    selected_proba = cat_proba
    selected_model_name = "CatBoost"
elif "log_proba" in globals():
    selected_proba = log_proba
    selected_model_name = "Logistic Regression"
else:
    raise ValueError("Hech qanday probability topilmadi: xgb_proba, cat_proba yoki log_proba kerak.")

alert_df = X_test.copy()
alert_df["true_serious_ade"] = y_test.values
alert_df["predicted_risk"] = selected_proba

def assign_alert_tier(p):
    if p >= 0.80:
        return "Critical"
    elif p >= 0.50:
        return "High"
    elif p >= 0.20:
        return "Advisory"
    else:
        return "Suppressed low-risk"

alert_df["alert_tier"] = alert_df["predicted_risk"].apply(assign_alert_tier)

alert_summary = alert_df.groupby("alert_tier").agg(
    total_alerts=("alert_tier", "count"),
    serious_ade_cases=("true_serious_ade", "sum"),
    mean_predicted_risk=("predicted_risk", "mean")
).reset_index()

alert_summary["percent_of_alerts"] = alert_summary["total_alerts"] / len(alert_df) * 100
alert_summary["serious_ade_rate"] = alert_summary["serious_ade_cases"] / alert_summary["total_alerts"] * 100

# Tartib bilan chiqarish
tier_order = ["Critical", "High", "Advisory", "Suppressed low-risk"]
alert_summary["tier_order"] = alert_summary["alert_tier"].map({t: i for i, t in enumerate(tier_order)})
alert_summary = alert_summary.sort_values("tier_order").drop(columns=["tier_order"])

print(f"\n=== alert_summary jadvali | Model: {selected_model_name} ===")
display(alert_summary)

alert_summary.to_csv("alert_tier_summary.csv", index=False)
print("Saved: alert_tier_summary.csv")


# =========================
# 3) alert_strategy_df natijasi
# =========================

total_alerts = len(alert_df)

active_alerts = alert_df[alert_df["alert_tier"].isin(["Critical", "High"])].shape[0]
suppressed_or_noninterruptive = total_alerts - active_alerts

alert_burden_reduction = suppressed_or_noninterruptive / total_alerts * 100

serious_total = alert_df["true_serious_ade"].sum()
serious_captured_active = alert_df[
    (alert_df["alert_tier"].isin(["Critical", "High"])) &
    (alert_df["true_serious_ade"] == 1)
].shape[0]

serious_sensitivity_retained = (
    serious_captured_active / serious_total * 100
    if serious_total > 0 else np.nan
)

alert_strategy_df = pd.DataFrame([{
    "Selected model": selected_model_name,
    "Strategy": "AI-guided triage: Critical + High shown actively",
    "Total potential alerts": total_alerts,
    "Active alerts shown": active_alerts,
    "Suppressed/non-interruptive alerts": suppressed_or_noninterruptive,
    "Alert burden reduction (%)": alert_burden_reduction,
    "Total serious ADE cases": serious_total,
    "Serious ADE captured in active alerts": serious_captured_active,
    "Serious ADE sensitivity retained (%)": serious_sensitivity_retained
}])

print("\n=== alert_strategy_df natijasi ===")
display(alert_strategy_df)

alert_strategy_df.to_csv("alert_strategy_summary.csv", index=False)
print("Saved: alert_strategy_summary.csv")

In [ ]:
threshold_results = []

# Critical threshold doim yuqori, High thresholdni o‘zgartirib ko‘ramiz
for high_threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    active = alert_df["predicted_risk"] >= high_threshold
    
    total_alerts = len(alert_df)
    active_alerts = active.sum()
    suppressed = total_alerts - active_alerts
    
    burden_reduction = suppressed / total_alerts * 100
    
    serious_total = alert_df["true_serious_ade"].sum()
    serious_captured = alert_df.loc[active & (alert_df["true_serious_ade"] == 1)].shape[0]
    sensitivity_retained = serious_captured / serious_total * 100
    
    false_positive_active = alert_df.loc[active & (alert_df["true_serious_ade"] == 0)].shape[0]
    ppv_active = serious_captured / active_alerts * 100 if active_alerts > 0 else np.nan
    
    threshold_results.append({
        "Active alert threshold": high_threshold,
        "Total potential alerts": total_alerts,
        "Active alerts shown": active_alerts,
        "Suppressed/non-interruptive alerts": suppressed,
        "Alert burden reduction (%)": burden_reduction,
        "Serious ADE sensitivity retained (%)": sensitivity_retained,
        "Active alert PPV (%)": ppv_active,
        "False positive active alerts": false_positive_active
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df

In [ ]:
threshold_results_df.to_csv("threshold_optimization_results.csv", index=False)
print("Saved: threshold_optimization_results.csv")

In [ ]:
import shap
import matplotlib.pyplot as plt

print("SHAP version:", shap.__version__)

In [ ]:
xgb_preprocessor = xgb_model.named_steps["preprocessor"]
xgb_estimator = xgb_model.named_steps["model"]

print("Preprocessor:")
print(xgb_preprocessor)

print("\nXGBoost estimator:")
print(xgb_estimator)

In [ ]:
shap_sample_size = 2000

X_shap = X_test.sample(
    n=min(shap_sample_size, len(X_test)),
    random_state=42
)

X_shap_transformed = xgb_preprocessor.transform(X_shap)

print("Original SHAP sample:", X_shap.shape)
print("Transformed SHAP sample:", X_shap_transformed.shape)

In [ ]:
numeric_feature_names = numeric_cols

cat_encoder = xgb_preprocessor.named_transformers_["cat"].named_steps["onehot"]
categorical_feature_names = cat_encoder.get_feature_names_out(categorical_cols).tolist()

feature_names = numeric_feature_names + categorical_feature_names

print("Feature names count:", len(feature_names))
print("Transformed columns:", X_shap_transformed.shape[1])

print("\nFirst 30 feature names:")
feature_names[:30]

In [ ]:
explainer = shap.TreeExplainer(xgb_estimator)
shap_values = explainer.shap_values(X_shap_transformed)

print("SHAP values shape:", np.array(shap_values).shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
shap.summary_plot(
    shap_values,
    X_shap_transformed,
    feature_names=feature_names,
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: shap_summary_plot.png")

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_abs_SHAP": mean_abs_shap
}).sort_values("Mean_abs_SHAP", ascending=False)

top15_shap = shap_importance_df.head(15).copy()
top15_shap["Rank"] = range(1, len(top15_shap) + 1)
top15_shap = top15_shap[["Rank", "Feature", "Mean_abs_SHAP"]]

display(top15_shap)

top15_shap.to_csv("top15_shap_features.csv", index=False)
print("Saved: top15_shap_features.csv")

In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_abs_SHAP": mean_abs_shap
}).sort_values("Mean_abs_SHAP", ascending=False)

top15_shap = shap_importance_df.head(15).copy()
top15_shap["Rank"] = range(1, len(top15_shap) + 1)
top15_shap = top15_shap[["Rank", "Feature", "Mean_abs_SHAP"]]

display(top15_shap)

top15_shap.to_csv("top15_shap_features.csv", index=False)
print("Saved: top15_shap_features.csv")

In [ ]:
plt.figure()
shap.summary_plot(
    shap_values,
    X_shap_transformed,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("shap_feature_importance_bar.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: shap_feature_importance_bar.png")

In [ ]:
plt.figure()
shap.summary_plot(
    shap_values,
    X_shap_transformed,
    feature_names=feature_names,
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("shap_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: shap_summary_plot.png")

In [ ]:
high_risk_index = np.argmax(xgb_proba)

high_risk_case = X_test.iloc[[high_risk_index]]
high_risk_true = y_test.iloc[high_risk_index]
high_risk_prob = xgb_proba[high_risk_index]

print("High-risk predicted probability:", high_risk_prob)
print("True serious ADE:", high_risk_true)

display(high_risk_case)

In [ ]:
high_risk_transformed = xgb_preprocessor.transform(high_risk_case)
high_risk_shap = explainer.shap_values(high_risk_transformed)

case_shap_df = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_value": high_risk_shap[0]
})

case_shap_df["Abs_SHAP"] = case_shap_df["SHAP_value"].abs()
case_shap_df = case_shap_df.sort_values("Abs_SHAP", ascending=False)

top_local_shap = case_shap_df.head(15)
display(top_local_shap)

top_local_shap.to_csv("local_high_risk_shap_explanation.csv", index=False)
print("Saved: local_high_risk_shap_explanation.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    brier_score_loss
)

fairness_df = X_test.copy()
fairness_df["true_serious_ade"] = y_test.values
fairness_df["predicted_risk"] = xgb_proba
fairness_df["predicted_label"] = (xgb_proba >= 0.5).astype(int)

print("Fairness dataframe shape:", fairness_df.shape)
fairness_df.head()

In [ ]:
def subgroup_metrics(data, subgroup_col, min_group_size=100):
    rows = []

    if subgroup_col not in data.columns:
        print(f"{subgroup_col} ustuni topilmadi.")
        return pd.DataFrame()

    for group, g in data.groupby(subgroup_col, dropna=False):
        if len(g) < min_group_size:
            continue

        y_true_g = g["true_serious_ade"]
        y_prob_g = g["predicted_risk"]
        y_pred_g = g["predicted_label"]

        # AUROC faqat 0 va 1 ikkalasi bo‘lsa hisoblanadi
        if y_true_g.nunique() < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(y_true_g, y_prob_g)
            auprc = average_precision_score(y_true_g, y_prob_g)

        tn, fp, fn, tp = confusion_matrix(y_true_g, y_pred_g, labels=[0, 1]).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
        f1 = f1_score(y_true_g, y_pred_g, zero_division=0)
        brier = brier_score_loss(y_true_g, y_prob_g)

        rows.append({
            "Subgroup variable": subgroup_col,
            "Subgroup": group,
            "n": len(g),
            "ADE rate (%)": y_true_g.mean() * 100,
            "AUROC": auroc,
            "AUPRC": auprc,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "PPV": ppv,
            "NPV": npv,
            "F1": f1,
            "FNR": fnr,
            "FPR": fpr,
            "Brier Score": brier
        })

    result = pd.DataFrame(rows)

    if len(result) > 0:
        result["AUROC gap vs best"] = result["AUROC"].max() - result["AUROC"]
        result["Sensitivity gap vs best"] = result["Sensitivity"].max() - result["Sensitivity"]
        result["FNR gap vs lowest"] = result["FNR"] - result["FNR"].min()

    return result

In [ ]:
fairness_sex = subgroup_metrics(fairness_df, "patient_sex", min_group_size=100)
display(fairness_sex)

fairness_sex.to_csv("fairness_by_patient_sex.csv", index=False)
print("Saved: fairness_by_patient_sex.csv")

In [ ]:
fairness_age = subgroup_metrics(fairness_df, "age_group", min_group_size=100)
display(fairness_age)

fairness_age.to_csv("fairness_by_age_group.csv", index=False)
print("Saved: fairness_by_age_group.csv")

In [ ]:
top_countries = fairness_df["country"].value_counts().head(10).index

fairness_country_df = fairness_df[fairness_df["country"].isin(top_countries)].copy()

fairness_country = subgroup_metrics(fairness_country_df, "country", min_group_size=100)
display(fairness_country)

fairness_country.to_csv("fairness_by_country_top10.csv", index=False)
print("Saved: fairness_by_country_top10.csv")

In [ ]:
fairness_drug_count = subgroup_metrics(fairness_df, "drug_count_category", min_group_size=100)
display(fairness_drug_count)

fairness_drug_count.to_csv("fairness_by_drug_count_category.csv", index=False)
print("Saved: fairness_by_drug_count_category.csv")

In [ ]:
fairness_tables = []

for table in [fairness_sex, fairness_age, fairness_country, fairness_drug_count]:
    if table is not None and len(table) > 0:
        fairness_tables.append(table)

fairness_all = pd.concat(fairness_tables, ignore_index=True)

display(fairness_all)

fairness_all.to_csv("fairness_all_subgroups.csv", index=False)
print("Saved: fairness_all_subgroups.csv")

In [ ]:
fairness_all["Fairness flag"] = "No major concern"

fairness_all.loc[
    fairness_all["AUROC gap vs best"] > 0.05,
    "Fairness flag"
] = "AUROC gap > 0.05"

fairness_all.loc[
    fairness_all["Sensitivity gap vs best"] > 0.10,
    "Fairness flag"
] = "Sensitivity gap > 0.10"

fairness_all.loc[
    fairness_all["FNR gap vs lowest"] > 0.10,
    "Fairness flag"
] = "FNR gap > 0.10"

display(fairness_all)

fairness_all.to_csv("fairness_all_subgroups_with_flags.csv", index=False)
print("Saved: fairness_all_subgroups_with_flags.csv")

In [ ]:
fairness_summary = fairness_all[[
    "Subgroup variable",
    "Subgroup",
    "n",
    "ADE rate (%)",
    "AUROC",
    "Sensitivity",
    "Specificity",
    "FNR",
    "FPR",
    "Brier Score",
    "Fairness flag"
]].copy()

display(fairness_summary)

fairness_summary.to_csv("fairness_summary_for_manuscript.csv", index=False)
print("Saved: fairness_summary_for_manuscript.csv")

In [ ]:
print("=== FAIRNESS SUMMARY ===")

for subgroup_var in fairness_summary["Subgroup variable"].unique():
    sub = fairness_summary[fairness_summary["Subgroup variable"] == subgroup_var]
    
    print(f"\nSubgroup variable: {subgroup_var}")
    print("Groups:", sub["Subgroup"].tolist())
    print("AUROC range:", round(sub["AUROC"].min(), 3), "-", round(sub["AUROC"].max(), 3))
    print("Sensitivity range:", round(sub["Sensitivity"].min(), 3), "-", round(sub["Sensitivity"].max(), 3))
    print("FNR range:", round(sub["FNR"].min(), 3), "-", round(sub["FNR"].max(), 3))
    
    flags = sub[sub["Fairness flag"] != "No major concern"]
    if len(flags) == 0:
        print("No major fairness concern by predefined thresholds.")
    else:
        print("Potential fairness concerns:")
        display(flags[["Subgroup", "Fairness flag"]])

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    brier_score_loss
)

# =========================
# 1) Fairness dataframe
# =========================

fairness_df = X_test.copy()
fairness_df["true_serious_ade"] = y_test.values
fairness_df["predicted_risk"] = xgb_proba
fairness_df["predicted_label"] = (xgb_proba >= 0.5).astype(int)

print("Fairness dataframe shape:", fairness_df.shape)
display(fairness_df.head())


# =========================
# 2) Subgroup metric function
# =========================

def subgroup_metrics(data, subgroup_col, min_group_size=100):
    rows = []

    if subgroup_col not in data.columns:
        print(f"Ustun topilmadi: {subgroup_col}")
        return pd.DataFrame()

    for group, g in data.groupby(subgroup_col, dropna=False):
        if len(g) < min_group_size:
            continue

        y_true_g = g["true_serious_ade"]
        y_prob_g = g["predicted_risk"]
        y_pred_g = g["predicted_label"]

        if y_true_g.nunique() < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(y_true_g, y_prob_g)
            auprc = average_precision_score(y_true_g, y_prob_g)

        tn, fp, fn, tp = confusion_matrix(
            y_true_g,
            y_pred_g,
            labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
        f1 = f1_score(y_true_g, y_pred_g, zero_division=0)
        brier = brier_score_loss(y_true_g, y_prob_g)

        rows.append({
            "Subgroup variable": subgroup_col,
            "Subgroup": group,
            "n": len(g),
            "ADE rate (%)": y_true_g.mean() * 100,
            "AUROC": auroc,
            "AUPRC": auprc,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "PPV": ppv,
            "NPV": npv,
            "F1": f1,
            "FNR": fnr,
            "FPR": fpr,
            "Brier Score": brier
        })

    result = pd.DataFrame(rows)

    if len(result) > 0:
        result["AUROC gap vs best"] = result["AUROC"].max() - result["AUROC"]
        result["Sensitivity gap vs best"] = result["Sensitivity"].max() - result["Sensitivity"]
        result["FNR gap vs lowest"] = result["FNR"] - result["FNR"].min()

    return result


# =========================
# 3) Run subgroup analyses
# =========================

fairness_tables = []

# patient_sex
if "patient_sex" in fairness_df.columns:
    fairness_sex = subgroup_metrics(fairness_df, "patient_sex", min_group_size=100)
    fairness_tables.append(fairness_sex)
    print("\n=== Fairness by patient_sex ===")
    display(fairness_sex)
else:
    print("patient_sex ustuni topilmadi.")

# age_group
if "age_group" in fairness_df.columns:
    fairness_age = subgroup_metrics(fairness_df, "age_group", min_group_size=100)
    fairness_tables.append(fairness_age)
    print("\n=== Fairness by age_group ===")
    display(fairness_age)
else:
    print("age_group ustuni topilmadi.")

# country top 10
if "country" in fairness_df.columns:
    top_countries = fairness_df["country"].value_counts().head(10).index
    fairness_country_df = fairness_df[fairness_df["country"].isin(top_countries)].copy()
    fairness_country = subgroup_metrics(fairness_country_df, "country", min_group_size=100)
    fairness_tables.append(fairness_country)
    print("\n=== Fairness by country top 10 ===")
    display(fairness_country)
else:
    print("country ustuni topilmadi.")

# drug_count_category
if "drug_count_category" in fairness_df.columns:
    fairness_drug_count = subgroup_metrics(fairness_df, "drug_count_category", min_group_size=100)
    fairness_tables.append(fairness_drug_count)
    print("\n=== Fairness by drug_count_category ===")
    display(fairness_drug_count)
else:
    print("drug_count_category ustuni topilmadi.")


# =========================
# 4) Combine all fairness tables
# =========================

fairness_tables = [t for t in fairness_tables if t is not None and len(t) > 0]

if len(fairness_tables) == 0:
    raise ValueError("Fairness jadvali yaratilmadi. Kerakli subgroup ustunlari topilmadi.")

fairness_all = pd.concat(fairness_tables, ignore_index=True)


# =========================
# 5) Add fairness flags
# =========================

fairness_all["Fairness flag"] = "No major concern"

fairness_all.loc[
    fairness_all["AUROC gap vs best"] > 0.05,
    "Fairness flag"
] = "AUROC gap > 0.05"

fairness_all.loc[
    fairness_all["Sensitivity gap vs best"] > 0.10,
    "Fairness flag"
] = "Sensitivity gap > 0.10"

fairness_all.loc[
    fairness_all["FNR gap vs lowest"] > 0.10,
    "Fairness flag"
] = "FNR gap > 0.10"


# =========================
# 6) Manuscript-ready summary table
# =========================

fairness_summary = fairness_all[[
    "Subgroup variable",
    "Subgroup",
    "n",
    "ADE rate (%)",
    "AUROC",
    "AUPRC",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
    "FNR",
    "FPR",
    "Brier Score",
    "Fairness flag"
]].copy()

# Raqamlarni o‘qishga qulay qilish
numeric_cols_to_round = [
    "ADE rate (%)", "AUROC", "AUPRC", "Sensitivity", "Specificity",
    "PPV", "NPV", "F1", "FNR", "FPR", "Brier Score"
]

for col in numeric_cols_to_round:
    if col in fairness_summary.columns:
        fairness_summary[col] = fairness_summary[col].round(4)

print("\n=== fairness_summary jadvali ===")
display(fairness_summary)

fairness_summary.to_csv("fairness_summary_for_manuscript.csv", index=False)
print("Saved: fairness_summary_for_manuscript.csv")


# =========================
# 7) Automatic FAIRNESS SUMMARY
# =========================

print("\n\n=== FAIRNESS SUMMARY ===")

for subgroup_var in fairness_summary["Subgroup variable"].unique():
    sub = fairness_summary[fairness_summary["Subgroup variable"] == subgroup_var]

    print(f"\nSubgroup variable: {subgroup_var}")
    print("Groups:", sub["Subgroup"].astype(str).tolist())

    print(
        "AUROC range:",
        round(sub["AUROC"].min(), 3),
        "-",
        round(sub["AUROC"].max(), 3)
    )

    print(
        "Sensitivity range:",
        round(sub["Sensitivity"].min(), 3),
        "-",
        round(sub["Sensitivity"].max(), 3)
    )

    print(
        "FNR range:",
        round(sub["FNR"].min(), 3),
        "-",
        round(sub["FNR"].max(), 3)
    )

    flags = sub[sub["Fairness flag"] != "No major concern"]

    if len(flags) == 0:
        print("No major fairness concern by predefined thresholds.")
    else:
        print("Potential fairness concerns:")
        display(flags[[
            "Subgroup",
            "AUROC",
            "Sensitivity",
            "FNR",
            "Fairness flag"
        ]])


# =========================
# 8) Save all detailed tables
# =========================

fairness_all.to_csv("fairness_all_subgroups_with_flags.csv", index=False)
print("\nSaved: fairness_all_subgroups_with_flags.csv")

In [ ]:
# =========================
# Leakage sensitivity setup
# =========================

leakage_features = [
    "reaction_outcomes"
]

feature_cols_leakage_reduced = [
    c for c in feature_cols
    if c not in leakage_features
]

print("Original feature count:", len(feature_cols))
print("Leakage-reduced feature count:", len(feature_cols_leakage_reduced))
print("Removed leakage-risk features:", leakage_features)

X_lr = model_df[feature_cols_leakage_reduced]
y_lr = model_df["serious_ade"].astype(int)

print("X leakage-reduced shape:", X_lr.shape)
print("Target distribution:")
print(y_lr.value_counts(normalize=True))

In [ ]:
print("model_df mavjudmi?", "model_df" in globals())
print("df mavjudmi?", "df" in globals())
print("DATA_PATH mavjudmi?", "DATA_PATH" in globals())

In [ ]:
feature_cols = [c for c in model_df.columns if c != "serious_ade"]

print("Feature count:", len(feature_cols))
print("Feature columns:")
for c in feature_cols:
    print("-", c)

In [ ]:
reaction_outcome_cols = [
    c for c in model_df.columns 
    if "reaction" in c or "outcome" in c
]

print("Reaction/outcome bilan bog‘liq ustunlar:")
for c in reaction_outcome_cols:
    print("-", c)

In [ ]:
reaction_outcome_cols = [
    c for c in model_df.columns 
    if "reaction" in c or "outcome" in c
]

print("Reaction/outcome bilan bog‘liq ustunlar:")
for c in reaction_outcome_cols:
    print("-", c)

In [ ]:
from sklearn.model_selection import train_test_split

X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr,
    y_lr,
    test_size=0.2,
    random_state=42,
    stratify=y_lr
)

print("Train:", X_lr_train.shape)
print("Test:", X_lr_test.shape)

In [ ]:
# =========================
# Re-create feature_cols + X_lr/y_lr
# =========================

# 1) feature_cols ni model_df ichidan qayta yaratamiz
feature_cols = [c for c in model_df.columns if c != "serious_ade"]

print("Feature count:", len(feature_cols))
print("Feature columns:")
for c in feature_cols:
    print("-", c)

# 2) reaction/outcome bilan bog‘liq ustunlarni ko‘ramiz
reaction_outcome_cols = [
    c for c in feature_cols 
    if "reaction" in c.lower() or "outcome" in c.lower()
]

print("\nReaction/outcome bilan bog‘liq ustunlar:")
for c in reaction_outcome_cols:
    print("-", c)

# 3) leakage-risk ustunlarni aniqlaymiz
# Asosiy xavf: reaction_outcomes
leakage_features = []

if "reaction_outcomes" in feature_cols:
    leakage_features.append("reaction_outcomes")

# Agar boshqa outcome nomli ustunlar bo‘lsa, ularni ham ko‘rib chiqamiz
# Lekin reactions, primary_reaction, num_reactions qoladi.
for c in feature_cols:
    c_low = c.lower()
    if "outcome" in c_low and c not in leakage_features:
        leakage_features.append(c)

print("\nOlib tashlanadigan leakage-risk features:")
print(leakage_features)

# 4) leakage-reduced feature list
feature_cols_leakage_reduced = [
    c for c in feature_cols
    if c not in leakage_features
]

print("\nOriginal feature count:", len(feature_cols))
print("Leakage-reduced feature count:", len(feature_cols_leakage_reduced))

# 5) X_lr va y_lr yaratamiz
X_lr = model_df[feature_cols_leakage_reduced].copy()
y_lr = model_df["serious_ade"].astype(int).copy()

print("\nX_lr shape:", X_lr.shape)
print("y_lr distribution:")
print(y_lr.value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr,
    y_lr,
    test_size=0.2,
    random_state=42,
    stratify=y_lr
)

print("Train:", X_lr_train.shape)
print("Test:", X_lr_test.shape)

In [ ]:
print("df shape:", df.shape)
print("model_df shape:", model_df.shape)

print("\ndf columns:")
print(df.columns.tolist())

print("\nmodel_df columns:")
print(model_df.columns.tolist())

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()

csv_files = list(BASE_DIR.glob("*.csv"))

print("Topilgan CSV fayllar:")
for i, f in enumerate(csv_files, start=1):
    print(i, f.name)

In [ ]:
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()

all_csv = list(BASE_DIR.glob("*.csv"))

# Natija fayllarini chiqarib tashlaymiz
exclude_keywords = [
    "model_results",
    "alert",
    "threshold",
    "fairness",
    "shap",
    "leakage",
    "temporal",
    "summary"
]

candidate_csv = [
    f for f in all_csv
    if not any(k in f.name.lower() for k in exclude_keywords)
]

print("Original database bo‘lishi mumkin bo‘lgan fayllar:")
for i, f in enumerate(candidate_csv, start=1):
    print(i, f.name)

DATA_PATH = candidate_csv[0]
print("\nTanlangan original dataset:", DATA_PATH.name)

In [ ]:
df = pd.read_csv(DATA_PATH, nrows=200000, low_memory=False)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(".", "_")
)

print("df shape:", df.shape)
print("\nColumns:")
for c in df.columns:
    print("-", c)

df.head()

In [ ]:
print("serious_ade bormi?", "serious_ade" in df.columns)

if "serious_ade" in df.columns:
    df["serious_ade"] = pd.to_numeric(df["serious_ade"], errors="coerce").fillna(0).astype(int)
else:
    possible_label_cols = [
        c for c in df.columns
        if any(word in c for word in [
            "serious", "death", "hospital", "life_threat", "lifethreat",
            "disab", "congen", "fatal"
        ])
    ]

    print("Possible label columns:", possible_label_cols)

    for c in possible_label_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    df["serious_ade"] = (df[possible_label_cols].sum(axis=1) > 0).astype(int)

print("\nTarget distribution:")
print(df["serious_ade"].value_counts())
print(df["serious_ade"].value_counts(normalize=True))

In [ ]:
candidate_feature_cols = [
    "year",
    "num_reactions",
    "num_drugs",
    "patient_age_years",
    "patient_weight_kg",
    "report_age_days",
    "reactions",
    "primary_reaction",
    "reaction_outcomes",
    "suspect_drug",
    "drug_route",
    "drug_indication",
    "manufacturer",
    "drug_count_category",
    "age_group",
    "patient_sex",
    "country"
]

feature_cols = [c for c in candidate_feature_cols if c in df.columns]

print("Selected feature columns:")
for c in feature_cols:
    print("-", c)

model_df = df[feature_cols + ["serious_ade"]].copy()

print("\nmodel_df shape:", model_df.shape)
print("Target distribution:")
print(model_df["serious_ade"].value_counts(normalize=True))

In [ ]:
leakage_features = ["reaction_outcomes"]

feature_cols_leakage_reduced = [
    c for c in feature_cols
    if c not in leakage_features
]

print("Original feature count:", len(feature_cols))
print("Leakage-reduced feature count:", len(feature_cols_leakage_reduced))
print("Removed:", leakage_features)

X_lr = model_df[feature_cols_leakage_reduced].copy()
y_lr = model_df["serious_ade"].astype(int).copy()

print("X_lr shape:", X_lr.shape)
print("y_lr shape:", y_lr.shape)
print(y_lr.value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split

X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr,
    y_lr,
    test_size=0.2,
    random_state=42,
    stratify=y_lr
)

print("Train:", X_lr_train.shape)
print("Test:", X_lr_test.shape)

print("\nTrain target:")
print(y_lr_train.value_counts(normalize=True))

print("\nTest target:")
print(y_lr_test.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)
import pandas as pd
import numpy as np

categorical_cols_lr = X_lr_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_lr = X_lr_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical:", categorical_cols_lr)
print("Numeric:", numeric_cols_lr)

numeric_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_cols_lr),
        ("cat", categorical_transformer_lr, categorical_cols_lr)
    ]
)

xgb_model_lr = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model_lr.fit(X_lr_train, y_lr_train)

xgb_lr_proba = xgb_model_lr.predict_proba(X_lr_test)[:, 1]
xgb_lr_pred = (xgb_lr_proba >= 0.5).astype(int)

print("Leakage-reduced XGBoost AUROC:", roc_auc_score(y_lr_test, xgb_lr_proba))
print("Leakage-reduced XGBoost AUPRC:", average_precision_score(y_lr_test, xgb_lr_proba))
print(classification_report(y_lr_test, xgb_lr_pred))
print(confusion_matrix(y_lr_test, xgb_lr_pred))

In [ ]:
def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

leakage_reduced_metrics = get_metrics(
    "Leakage-reduced XGBoost without reaction_outcomes",
    y_lr_test,
    xgb_lr_proba
)

# Avvalgi full model natijalari, siz oldin olgan jadvaldan
full_model_metrics = {
    "Model": "Full XGBoost with reaction_outcomes",
    "AUROC": 0.865324,
    "AUPRC": 0.841289,
    "Sensitivity": 0.738570,
    "Specificity": 0.808544,
    "PPV": 0.761037,
    "NPV": 0.789307,
    "F1": 0.749635,
    "Brier Score": 0.149629
}

leakage_comparison_df = pd.DataFrame([
    full_model_metrics,
    leakage_reduced_metrics
])

display(leakage_comparison_df)

leakage_comparison_df.to_csv("leakage_sensitivity_analysis.csv", index=False)
print("Saved: leakage_sensitivity_analysis.csv")

In [ ]:
print("year column exists:", "year" in model_df.columns)

model_df["year"] = pd.to_numeric(model_df["year"], errors="coerce")

print(model_df["year"].value_counts(dropna=False).sort_index())
print(model_df["year"].describe())

In [ ]:
temporal_df = model_df.copy()
temporal_df["year"] = pd.to_numeric(temporal_df["year"], errors="coerce")

train_temporal = temporal_df[
    (temporal_df["year"] >= 2015) & (temporal_df["year"] <= 2021)
].copy()

valid_temporal = temporal_df[
    (temporal_df["year"] >= 2022) & (temporal_df["year"] <= 2023)
].copy()

test_temporal = temporal_df[
    (temporal_df["year"] >= 2024) & (temporal_df["year"] <= 2026)
].copy()

print("Train temporal:", train_temporal.shape)
print("Validation temporal:", valid_temporal.shape)
print("Test temporal:", test_temporal.shape)

print("\nTrain target distribution:")
print(train_temporal["serious_ade"].value_counts(normalize=True))

print("\nValidation target distribution:")
print(valid_temporal["serious_ade"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_temporal["serious_ade"].value_counts(normalize=True))

In [ ]:
train_temporal = temporal_df[
    (temporal_df["year"] >= 2015) & (temporal_df["year"] <= 2020)
].copy()

valid_temporal = temporal_df[
    (temporal_df["year"] >= 2021) & (temporal_df["year"] <= 2022)
].copy()

test_temporal = temporal_df[
    (temporal_df["year"] >= 2023) & (temporal_df["year"] <= 2026)
].copy()

print("Train temporal:", train_temporal.shape)
print("Validation temporal:", valid_temporal.shape)
print("Test temporal:", test_temporal.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)
import pandas as pd
import numpy as np

temporal_feature_cols = [
    c for c in feature_cols
    if c not in ["reaction_outcomes"]
]

X_train_t = train_temporal[temporal_feature_cols]
y_train_t = train_temporal["serious_ade"].astype(int)

X_valid_t = valid_temporal[temporal_feature_cols]
y_valid_t = valid_temporal["serious_ade"].astype(int)

X_test_t = test_temporal[temporal_feature_cols]
y_test_t = test_temporal["serious_ade"].astype(int)

categorical_cols_t = X_train_t.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_t = X_train_t.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical columns:", categorical_cols_t)
print("Numeric columns:", numeric_cols_t)

numeric_transformer_t = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_t = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_t = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_t, numeric_cols_t),
        ("cat", categorical_transformer_t, categorical_cols_t)
    ]
)

xgb_temporal_model = Pipeline(steps=[
    ("preprocessor", preprocessor_t),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_temporal_model.fit(X_train_t, y_train_t)

valid_t_proba = xgb_temporal_model.predict_proba(X_valid_t)[:, 1]
test_t_proba = xgb_temporal_model.predict_proba(X_test_t)[:, 1]

In [ ]:
print("model_df shape:", model_df.shape)
print("year column exists:", "year" in model_df.columns)

model_df["year"] = pd.to_numeric(model_df["year"], errors="coerce")

year_counts = model_df["year"].value_counts(dropna=False).sort_index()
print(year_counts)

print("\nMin year:", model_df["year"].min())
print("Max year:", model_df["year"].max())

In [ ]:
temporal_df = model_df.copy()
temporal_df["year"] = pd.to_numeric(temporal_df["year"], errors="coerce")
temporal_df = temporal_df.dropna(subset=["year"]).copy()
temporal_df["year"] = temporal_df["year"].astype(int)

available_years = sorted(temporal_df["year"].unique())
print("Available years:", available_years)

if len(available_years) < 3:
    raise ValueError(
        f"Temporal validation uchun kamida 3 xil yil kerak. Sizda bor yillar: {available_years}"
    )

test_year = available_years[-1]
valid_year = available_years[-2]
train_years = available_years[:-2]

print("Train years:", train_years)
print("Validation year:", valid_year)
print("Test year:", test_year)

train_temporal = temporal_df[temporal_df["year"].isin(train_years)].copy()
valid_temporal = temporal_df[temporal_df["year"] == valid_year].copy()
test_temporal = temporal_df[temporal_df["year"] == test_year].copy()

print("\nTrain temporal:", train_temporal.shape)
print("Validation temporal:", valid_temporal.shape)
print("Test temporal:", test_temporal.shape)

print("\nTrain target distribution:")
print(train_temporal["serious_ade"].value_counts(normalize=True))

print("\nValidation target distribution:")
print(valid_temporal["serious_ade"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_temporal["serious_ade"].value_counts(normalize=True))

In [ ]:
temporal_df = model_df.copy()
temporal_df["year"] = pd.to_numeric(temporal_df["year"], errors="coerce")
temporal_df = temporal_df.dropna(subset=["year"]).copy()
temporal_df = temporal_df.sort_values("year").reset_index(drop=True)

n = len(temporal_df)

train_end = int(n * 0.60)
valid_end = int(n * 0.80)

train_temporal = temporal_df.iloc[:train_end].copy()
valid_temporal = temporal_df.iloc[train_end:valid_end].copy()
test_temporal = temporal_df.iloc[valid_end:].copy()

print("Train temporal:", train_temporal.shape)
print("Validation temporal:", valid_temporal.shape)
print("Test temporal:", test_temporal.shape)

print("\nTrain years:")
print(train_temporal["year"].value_counts().sort_index())

print("\nValidation years:")
print(valid_temporal["year"].value_counts().sort_index())

print("\nTest years:")
print(test_temporal["year"].value_counts().sort_index())

print("\nTrain target distribution:")
print(train_temporal["serious_ade"].value_counts(normalize=True))

print("\nValidation target distribution:")
print(valid_temporal["serious_ade"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_temporal["serious_ade"].value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)
import pandas as pd
import numpy as np

if len(train_temporal) == 0 or len(valid_temporal) == 0 or len(test_temporal) == 0:
    raise ValueError(
        f"Temporal split bo‘sh: train={len(train_temporal)}, "
        f"valid={len(valid_temporal)}, test={len(test_temporal)}. "
        "2-cell yoki 2B-cell bilan splitni qayta qiling."
    )

temporal_feature_cols = [
    c for c in feature_cols
    if c not in ["reaction_outcomes"]
]

X_train_t = train_temporal[temporal_feature_cols].copy()
y_train_t = train_temporal["serious_ade"].astype(int).copy()

X_valid_t = valid_temporal[temporal_feature_cols].copy()
y_valid_t = valid_temporal["serious_ade"].astype(int).copy()

X_test_t = test_temporal[temporal_feature_cols].copy()
y_test_t = test_temporal["serious_ade"].astype(int).copy()

categorical_cols_t = X_train_t.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_t = X_train_t.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Train:", X_train_t.shape)
print("Validation:", X_valid_t.shape)
print("Test:", X_test_t.shape)
print("Categorical columns:", categorical_cols_t)
print("Numeric columns:", numeric_cols_t)

numeric_transformer_t = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_t = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_t = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_t, numeric_cols_t),
        ("cat", categorical_transformer_t, categorical_cols_t)
    ]
)

xgb_temporal_model = Pipeline(steps=[
    ("preprocessor", preprocessor_t),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_temporal_model.fit(X_train_t, y_train_t)

valid_t_proba = xgb_temporal_model.predict_proba(X_valid_t)[:, 1]
test_t_proba = xgb_temporal_model.predict_proba(X_test_t)[:, 1]

print("Temporal model prediction completed.")

In [ ]:
def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

temporal_results_df = pd.DataFrame([
    get_metrics("Temporal validation set", y_valid_t, valid_t_proba),
    get_metrics("Temporal test set", y_test_t, test_t_proba)
])

display(temporal_results_df)

temporal_results_df.to_csv("temporal_validation_results.csv", index=False)
print("Saved: temporal_validation_results.csv")

In [ ]:
display(temporal_results_df)

temporal_results_df.to_csv("temporal_validation_results.csv", index=False)
print("Saved: temporal_validation_results.csv")

In [ ]:
import pandas as pd
import numpy as np

alert_lr_df = X_lr_test.copy()
alert_lr_df["true_serious_ade"] = y_lr_test.values
alert_lr_df["predicted_risk"] = xgb_lr_proba

threshold_results_lr = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    active = alert_lr_df["predicted_risk"] >= threshold
    
    total_alerts = len(alert_lr_df)
    active_alerts = int(active.sum())
    audit_queue = total_alerts - active_alerts
    
    burden_reduction = audit_queue / total_alerts * 100
    
    serious_total = int(alert_lr_df["true_serious_ade"].sum())
    serious_captured = int(alert_lr_df.loc[
        active & (alert_lr_df["true_serious_ade"] == 1)
    ].shape[0])
    
    sensitivity_retained = serious_captured / serious_total * 100 if serious_total > 0 else np.nan
    
    false_positive_active = int(alert_lr_df.loc[
        active & (alert_lr_df["true_serious_ade"] == 0)
    ].shape[0])
    
    ppv_active = serious_captured / active_alerts * 100 if active_alerts > 0 else np.nan
    
    threshold_results_lr.append({
        "Model": "Leakage-reduced XGBoost",
        "Active alert threshold": threshold,
        "Total potential alerts": total_alerts,
        "Active alerts shown": active_alerts,
        "Non-interruptive/audit queue": audit_queue,
        "Alert burden reduction (%)": burden_reduction,
        "Serious ADE sensitivity retained (%)": sensitivity_retained,
        "Active alert PPV (%)": ppv_active,
        "False positive active alerts": false_positive_active
    })

threshold_results_lr_df = pd.DataFrame(threshold_results_lr)

display(threshold_results_lr_df)

threshold_results_lr_df.to_csv("threshold_optimization_leakage_reduced.csv", index=False)
print("Saved: threshold_optimization_leakage_reduced.csv")

In [ ]:
def assign_alert_tier_safe(p):
    if p >= 0.80:
        return "Critical"
    elif p >= 0.50:
        return "High"
    elif p >= 0.30:
        return "Advisory"
    else:
        return "Low-risk audit queue"

alert_lr_df["alert_tier"] = alert_lr_df["predicted_risk"].apply(assign_alert_tier_safe)

alert_lr_summary = alert_lr_df.groupby("alert_tier").agg(
    total_alerts=("alert_tier", "count"),
    serious_ade_cases=("true_serious_ade", "sum"),
    mean_predicted_risk=("predicted_risk", "mean")
).reset_index()

alert_lr_summary["percent_of_alerts"] = (
    alert_lr_summary["total_alerts"] / len(alert_lr_df) * 100
)

alert_lr_summary["serious_ade_rate"] = (
    alert_lr_summary["serious_ade_cases"] /
    alert_lr_summary["total_alerts"] * 100
)

tier_order = ["Critical", "High", "Advisory", "Low-risk audit queue"]
alert_lr_summary["tier_order"] = alert_lr_summary["alert_tier"].map(
    {t: i for i, t in enumerate(tier_order)}
)

alert_lr_summary = (
    alert_lr_summary
    .sort_values("tier_order")
    .drop(columns=["tier_order"])
)

display(alert_lr_summary)

alert_lr_summary.to_csv("alert_tier_summary_leakage_reduced.csv", index=False)
print("Saved: alert_tier_summary_leakage_reduced.csv")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    brier_score_loss
)

fairness_lr_df = X_lr_test.copy()
fairness_lr_df["true_serious_ade"] = y_lr_test.values
fairness_lr_df["predicted_risk"] = xgb_lr_proba
fairness_lr_df["predicted_label"] = (xgb_lr_proba >= 0.5).astype(int)

def subgroup_metrics_safe(data, subgroup_col, min_group_size=100):
    rows = []

    if subgroup_col not in data.columns:
        print(f"Column not found: {subgroup_col}")
        return pd.DataFrame()

    for group, g in data.groupby(subgroup_col, dropna=False):
        if len(g) < min_group_size:
            continue

        y_true_g = g["true_serious_ade"]
        y_prob_g = g["predicted_risk"]
        y_pred_g = g["predicted_label"]

        if y_true_g.nunique() < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(y_true_g, y_prob_g)
            auprc = average_precision_score(y_true_g, y_prob_g)

        tn, fp, fn, tp = confusion_matrix(
            y_true_g,
            y_pred_g,
            labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
        f1 = f1_score(y_true_g, y_pred_g, zero_division=0)
        brier = brier_score_loss(y_true_g, y_prob_g)

        rows.append({
            "Subgroup variable": subgroup_col,
            "Subgroup": group,
            "n": len(g),
            "ADE rate (%)": y_true_g.mean() * 100,
            "AUROC": auroc,
            "AUPRC": auprc,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "PPV": ppv,
            "NPV": npv,
            "F1": f1,
            "FNR": fnr,
            "FPR": fpr,
            "Brier Score": brier
        })

    result = pd.DataFrame(rows)

    if len(result) > 0:
        result["AUROC gap vs best"] = result["AUROC"].max() - result["AUROC"]
        result["Sensitivity gap vs best"] = result["Sensitivity"].max() - result["Sensitivity"]
        result["FNR gap vs lowest"] = result["FNR"] - result["FNR"].min()

        result["Fairness flag"] = "No major concern"
        result.loc[
            result["AUROC gap vs best"] > 0.05,
            "Fairness flag"
        ] = "AUROC gap > 0.05"

        result.loc[
            result["Sensitivity gap vs best"] > 0.10,
            "Fairness flag"
        ] = "Sensitivity gap > 0.10"

        result.loc[
            result["FNR gap vs lowest"] > 0.10,
            "Fairness flag"
        ] = "FNR gap > 0.10"

    return result

fairness_lr_tables = []

for col in ["patient_sex", "age_group", "country", "drug_count_category"]:
    if col in fairness_lr_df.columns:
        if col == "country":
            top_countries = fairness_lr_df["country"].value_counts().head(10).index
            temp_df = fairness_lr_df[fairness_lr_df["country"].isin(top_countries)].copy()
            result = subgroup_metrics_safe(temp_df, col, min_group_size=100)
        else:
            result = subgroup_metrics_safe(fairness_lr_df, col, min_group_size=100)

        fairness_lr_tables.append(result)
        print(f"\n=== Fairness by {col} ===")
        display(result)

fairness_lr_all = pd.concat(
    [t for t in fairness_lr_tables if len(t) > 0],
    ignore_index=True
)

fairness_lr_summary = fairness_lr_all[[
    "Subgroup variable",
    "Subgroup",
    "n",
    "ADE rate (%)",
    "AUROC",
    "AUPRC",
    "Sensitivity",
    "Specificity",
    "FNR",
    "FPR",
    "Brier Score",
    "Fairness flag"
]].copy()

display(fairness_lr_summary)

fairness_lr_summary.to_csv("fairness_summary_leakage_reduced.csv", index=False)
print("Saved: fairness_summary_leakage_reduced.csv")

In [ ]:
from pathlib import Path

for f in Path.cwd().glob("*.csv"):
    if any(k in f.name.lower() for k in [
        "temporal",
        "leakage",
        "threshold",
        "alert_tier",
        "fairness"
    ]):
        print(f.name)

In [ ]:
from pathlib import Path
import pandas as pd

# Ko‘rish kerak bo‘lgan fayllar
target_files = {
    "temporal_results_df": "temporal_validation_results.csv",
    "threshold_results_lr_df": "threshold_optimization_leakage_reduced.csv",
    "alert_lr_summary": "alert_tier_summary_leakage_reduced.csv",
    "fairness_lr_summary": "fairness_summary_leakage_reduced.csv"
}

BASE_DIR = Path.cwd()

print("Current folder:", BASE_DIR)
print("\nCSV fayllarni tekshirish:")

loaded_tables = {}

for table_name, file_name in target_files.items():
    file_path = BASE_DIR / file_name
    
    print("\n" + "="*80)
    print(f"{table_name}  |  {file_name}")
    print("="*80)
    
    if file_path.exists():
        df_table = pd.read_csv(file_path)
        loaded_tables[table_name] = df_table
        
        print(f"Topildi: {file_path}")
        print("Shape:", df_table.shape)
        display(df_table)
    else:
        print(f"Topilmadi: {file_name}")
        print("Avval shu jadvalni yaratadigan analysis cellni run qilish kerak.")

# Jupyter xotirasiga ham qayta saqlab qo‘yamiz
if "temporal_results_df" in loaded_tables:
    temporal_results_df = loaded_tables["temporal_results_df"]

if "threshold_results_lr_df" in loaded_tables:
    threshold_results_lr_df = loaded_tables["threshold_results_lr_df"]

if "alert_lr_summary" in loaded_tables:
    alert_lr_summary = loaded_tables["alert_lr_summary"]

if "fairness_lr_summary" in loaded_tables:
    fairness_lr_summary = loaded_tables["fairness_lr_summary"]

print("\nTayyor. Topilgan jadvallar Jupyter xotirasiga ham yuklandi.")

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Leakage-reduced XGBoost pipeline ichidan preprocessor va modelni ajratamiz
xgb_lr_preprocessor = xgb_model_lr.named_steps["preprocessor"]
xgb_lr_estimator = xgb_model_lr.named_steps["model"]

print("Preprocessor:")
print(xgb_lr_preprocessor)

print("\nXGBoost estimator:")
print(xgb_lr_estimator)

In [ ]:
shap_sample_size = 2000

X_lr_shap = X_lr_test.sample(
    n=min(shap_sample_size, len(X_lr_test)),
    random_state=42
)

X_lr_shap_transformed = xgb_lr_preprocessor.transform(X_lr_shap)

print("Original SHAP sample:", X_lr_shap.shape)
print("Transformed SHAP sample:", X_lr_shap_transformed.shape)

In [ ]:
# Numeric feature names
numeric_feature_names_lr = numeric_cols_lr

# Categorical one-hot feature names
cat_encoder_lr = xgb_lr_preprocessor.named_transformers_["cat"].named_steps["onehot"]
categorical_feature_names_lr = cat_encoder_lr.get_feature_names_out(categorical_cols_lr).tolist()

feature_names_lr = numeric_feature_names_lr + categorical_feature_names_lr

print("Feature names count:", len(feature_names_lr))
print("Transformed columns:", X_lr_shap_transformed.shape[1])

print("\nFirst 30 feature names:")
feature_names_lr[:30]

In [ ]:
explainer_lr = shap.TreeExplainer(xgb_lr_estimator)
shap_values_lr = explainer_lr.shap_values(X_lr_shap_transformed)

print("SHAP values shape:", np.array(shap_values_lr).shape)

In [ ]:
plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_lr_shap_transformed,
    feature_names=feature_names_lr,
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("shap_summary_plot_leakage_reduced.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: shap_summary_plot_leakage_reduced.png")

In [ ]:
plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_lr_shap_transformed,
    feature_names=feature_names_lr,
    plot_type="bar",
    show=False,
    max_display=20
)

plt.tight_layout()
plt.savefig("shap_feature_importance_bar_leakage_reduced.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: shap_feature_importance_bar_leakage_reduced.png")

In [ ]:
mean_abs_shap_lr = np.abs(shap_values_lr).mean(axis=0)

shap_importance_lr_df = pd.DataFrame({
    "Feature": feature_names_lr,
    "Mean_abs_SHAP": mean_abs_shap_lr
}).sort_values("Mean_abs_SHAP", ascending=False)

top20_shap_lr = shap_importance_lr_df.head(20).copy()
top20_shap_lr["Rank"] = range(1, len(top20_shap_lr) + 1)
top20_shap_lr = top20_shap_lr[["Rank", "Feature", "Mean_abs_SHAP"]]

display(top20_shap_lr)

shap_importance_lr_df.to_csv("shap_feature_importance_leakage_reduced.csv", index=False)
top20_shap_lr.to_csv("top20_shap_features_leakage_reduced.csv", index=False)

print("Saved: shap_feature_importance_leakage_reduced.csv")
print("Saved: top20_shap_features_leakage_reduced.csv")

In [ ]:
high_risk_index_lr = np.argmax(xgb_lr_proba)

high_risk_case_lr = X_lr_test.iloc[[high_risk_index_lr]]
high_risk_true_lr = y_lr_test.iloc[high_risk_index_lr]
high_risk_prob_lr = xgb_lr_proba[high_risk_index_lr]

print("High-risk predicted probability:", high_risk_prob_lr)
print("True serious ADE:", high_risk_true_lr)

display(high_risk_case_lr)

In [ ]:
high_risk_transformed_lr = xgb_lr_preprocessor.transform(high_risk_case_lr)
high_risk_shap_lr = explainer_lr.shap_values(high_risk_transformed_lr)

case_shap_lr_df = pd.DataFrame({
    "Feature": feature_names_lr,
    "SHAP_value": high_risk_shap_lr[0]
})

case_shap_lr_df["Abs_SHAP"] = case_shap_lr_df["SHAP_value"].abs()
case_shap_lr_df = case_shap_lr_df.sort_values("Abs_SHAP", ascending=False)

top_local_shap_lr = case_shap_lr_df.head(15)

display(top_local_shap_lr)

top_local_shap_lr.to_csv("local_high_risk_shap_explanation_leakage_reduced.csv", index=False)

print("Saved: local_high_risk_shap_explanation_leakage_reduced.csv")

In [ ]:
import joblib

joblib.dump(xgb_model_lr, "xgboost_ade_model_leakage_reduced.pkl")

print("Saved: xgboost_ade_model_leakage_reduced.pkl")

In [ ]:
from pathlib import Path

important_outputs = [
    "leakage_sensitivity_analysis.csv",
    "temporal_validation_results.csv",
    "threshold_optimization_leakage_reduced.csv",
    "alert_tier_summary_leakage_reduced.csv",
    "fairness_summary_leakage_reduced.csv",
    "shap_summary_plot_leakage_reduced.png",
    "shap_feature_importance_bar_leakage_reduced.png",
    "top20_shap_features_leakage_reduced.csv",
    "local_high_risk_shap_explanation_leakage_reduced.csv",
    "xgboost_ade_model_leakage_reduced.pkl"
]

for fname in important_outputs:
    path = Path.cwd() / fname
    print(fname, "✅" if path.exists() else "❌")

In [ ]:
final_summary_rows = [
    {
        "Analysis block": "Full model performance",
        "Main finding": "Full XGBoost achieved AUROC 0.865 and AUPRC 0.841.",
        "Manuscript interpretation": "Strong internal held-out performance, but potential reaction_outcomes leakage required sensitivity analysis."
    },
    {
        "Analysis block": "Leakage sensitivity analysis",
        "Main finding": "After excluding reaction_outcomes, AUROC remained 0.862 and AUPRC 0.838.",
        "Manuscript interpretation": "Performance was not materially dependent on the potential outcome-proxy feature."
    },
    {
        "Analysis block": "Temporal validation",
        "Main finding": "Temporal test AUROC decreased to 0.800 and AUPRC to 0.788.",
        "Manuscript interpretation": "Temporal drift exists; periodic recalibration is required."
    },
    {
        "Analysis block": "Threshold optimisation",
        "Main finding": "Threshold 0.30 reduced alert burden by 36.9% while retaining 90.8% serious ADE sensitivity.",
        "Manuscript interpretation": "0.30 is the preferred safety-oriented threshold."
    },
    {
        "Analysis block": "Alert tier analysis",
        "Main finding": "Critical tier serious ADE rate was 92.9%; audit queue still contained 1,657 serious ADE cases.",
        "Manuscript interpretation": "Low-risk cases must not be deleted; use pharmacist audit queue."
    },
    {
        "Analysis block": "Fairness analysis",
        "Main finding": "Female, unknown-sex, infant, teen, unknown-age, and low-drug-count groups showed elevated FNR or lower AUROC.",
        "Manuscript interpretation": "Subgroup-specific calibration is required before deployment."
    },
    {
        "Analysis block": "SHAP explainability",
        "Main finding": "Top SHAP features should be interpreted after leakage-reduced analysis.",
        "Manuscript interpretation": "Explainability is valid at FAERS report level, not causal clinical mechanism level."
    }
]

final_summary_df = pd.DataFrame(final_summary_rows)

display(final_summary_df)

final_summary_df.to_csv("final_analysis_summary_for_manuscript.csv", index=False)

print("Saved: final_analysis_summary_for_manuscript.csv")

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()

# =========================
# 1) top20_shap_lr jadvali
# =========================

print("="*90)
print("1) top20_shap_lr jadvali")
print("="*90)

top20_file = BASE_DIR / "top20_shap_features_leakage_reduced.csv"

if top20_file.exists():
    top20_shap_lr = pd.read_csv(top20_file)
    print("Topildi:", top20_file.name)
    print("Shape:", top20_shap_lr.shape)
    display(top20_shap_lr)
else:
    print("Topilmadi: top20_shap_features_leakage_reduced.csv")
    print("Avval leakage-reduced SHAP analysis celllarini run qilish kerak.")


# =========================
# 2) top_local_shap_lr jadvali
# =========================

print("\n" + "="*90)
print("2) top_local_shap_lr jadvali")
print("="*90)

local_file = BASE_DIR / "local_high_risk_shap_explanation_leakage_reduced.csv"

if local_file.exists():
    top_local_shap_lr = pd.read_csv(local_file)
    print("Topildi:", local_file.name)
    print("Shape:", top_local_shap_lr.shape)
    display(top_local_shap_lr)
else:
    print("Topilmadi: local_high_risk_shap_explanation_leakage_reduced.csv")
    print("Avval high-risk local SHAP explanation cellini run qilish kerak.")


# =========================
# 3) final output check natijasi
# =========================

print("\n" + "="*90)
print("3) final output check natijasi")
print("="*90)

important_outputs = [
    "leakage_sensitivity_analysis.csv",
    "temporal_validation_results.csv",
    "threshold_optimization_leakage_reduced.csv",
    "alert_tier_summary_leakage_reduced.csv",
    "fairness_summary_leakage_reduced.csv",
    "shap_summary_plot_leakage_reduced.png",
    "shap_feature_importance_bar_leakage_reduced.png",
    "shap_feature_importance_leakage_reduced.csv",
    "top20_shap_features_leakage_reduced.csv",
    "local_high_risk_shap_explanation_leakage_reduced.csv",
    "xgboost_ade_model_leakage_reduced.pkl",
    "final_analysis_summary_for_manuscript.csv"
]

check_rows = []

for fname in important_outputs:
    path = BASE_DIR / fname
    check_rows.append({
        "File": fname,
        "Exists": "✅ Yes" if path.exists() else "❌ No",
        "Size KB": round(path.stat().st_size / 1024, 2) if path.exists() else None
    })

final_output_check = pd.DataFrame(check_rows)

display(final_output_check)

# Saqlab qo'yamiz
final_output_check.to_csv("final_output_check.csv", index=False)
print("Saved: final_output_check.csv")


# =========================
# Qo‘shimcha: barcha kerakli jadvallar xotiraga yuklandi
# =========================

print("\nTayyor:")
print("- top20_shap_lr")
print("- top_local_shap_lr")
print("- final_output_check")

In [ ]:
# =========================
# Strict leakage-reduced setup
# =========================

strict_leakage_features = [
    "reaction_outcomes",
    "primary_reaction",
    "reactions"
]

feature_cols_strict = [
    c for c in feature_cols
    if c not in strict_leakage_features
]

print("Original feature count:", len(feature_cols))
print("Strict leakage-reduced feature count:", len(feature_cols_strict))
print("Removed strict leakage-risk features:", strict_leakage_features)

X_strict = model_df[feature_cols_strict].copy()
y_strict = model_df["serious_ade"].astype(int).copy()

print("X_strict shape:", X_strict.shape)
print("y_strict shape:", y_strict.shape)

print("\nTarget distribution:")
print(y_strict.value_counts())
print(y_strict.value_counts(normalize=True))

print("\nStrict feature columns:")
for c in feature_cols_strict:
    print("-", c)

In [ ]:
from sklearn.model_selection import train_test_split

X_strict_train, X_strict_test, y_strict_train, y_strict_test = train_test_split(
    X_strict,
    y_strict,
    test_size=0.2,
    random_state=42,
    stratify=y_strict
)

print("Train:", X_strict_train.shape)
print("Test:", X_strict_test.shape)

print("\nTrain target:")
print(y_strict_train.value_counts(normalize=True))

print("\nTest target:")
print(y_strict_test.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)
import pandas as pd
import numpy as np

categorical_cols_strict = X_strict_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_strict = X_strict_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical:", categorical_cols_strict)
print("Numeric:", numeric_cols_strict)

numeric_transformer_strict = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_strict = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_strict = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_strict, numeric_cols_strict),
        ("cat", categorical_transformer_strict, categorical_cols_strict)
    ]
)

xgb_model_strict = Pipeline(steps=[
    ("preprocessor", preprocessor_strict),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model_strict.fit(X_strict_train, y_strict_train)

xgb_strict_proba = xgb_model_strict.predict_proba(X_strict_test)[:, 1]
xgb_strict_pred = (xgb_strict_proba >= 0.5).astype(int)

print("Strict leakage-reduced XGBoost AUROC:", roc_auc_score(y_strict_test, xgb_strict_proba))
print("Strict leakage-reduced XGBoost AUPRC:", average_precision_score(y_strict_test, xgb_strict_proba))
print(classification_report(y_strict_test, xgb_strict_pred))
print(confusion_matrix(y_strict_test, xgb_strict_pred))

In [ ]:
def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

full_model_metrics = {
    "Model": "Full XGBoost with reaction_outcomes",
    "AUROC": 0.865324,
    "AUPRC": 0.841289,
    "Sensitivity": 0.738570,
    "Specificity": 0.808544,
    "PPV": 0.761037,
    "NPV": 0.789307,
    "F1": 0.749635,
    "Brier Score": 0.149629
}

leakage_reduced_metrics = {
    "Model": "Leakage-reduced XGBoost without reaction_outcomes",
    "AUROC": 0.861959,
    "AUPRC": 0.837928,
    "Sensitivity": 0.731439,
    "Specificity": 0.808498,
    "PPV": 0.759224,
    "NPV": 0.784787,
    "F1": 0.745073,
    "Brier Score": 0.151460
}

strict_metrics = get_metrics(
    "Strict leakage-reduced XGBoost without reaction fields",
    y_strict_test,
    xgb_strict_proba
)

strict_leakage_comparison_df = pd.DataFrame([
    full_model_metrics,
    leakage_reduced_metrics,
    strict_metrics
])

display(strict_leakage_comparison_df)

strict_leakage_comparison_df.to_csv("strict_leakage_sensitivity_analysis.csv", index=False)
print("Saved: strict_leakage_sensitivity_analysis.csv")

In [ ]:
alert_strict_df = X_strict_test.copy()
alert_strict_df["true_serious_ade"] = y_strict_test.values
alert_strict_df["predicted_risk"] = xgb_strict_proba

threshold_results_strict = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    active = alert_strict_df["predicted_risk"] >= threshold
    
    total_alerts = len(alert_strict_df)
    active_alerts = int(active.sum())
    audit_queue = total_alerts - active_alerts
    
    burden_reduction = audit_queue / total_alerts * 100
    
    serious_total = int(alert_strict_df["true_serious_ade"].sum())
    serious_captured = int(alert_strict_df.loc[
        active & (alert_strict_df["true_serious_ade"] == 1)
    ].shape[0])
    
    sensitivity_retained = serious_captured / serious_total * 100 if serious_total > 0 else np.nan
    
    false_positive_active = int(alert_strict_df.loc[
        active & (alert_strict_df["true_serious_ade"] == 0)
    ].shape[0])
    
    ppv_active = serious_captured / active_alerts * 100 if active_alerts > 0 else np.nan
    
    threshold_results_strict.append({
        "Model": "Strict leakage-reduced XGBoost",
        "Active alert threshold": threshold,
        "Total potential alerts": total_alerts,
        "Active alerts shown": active_alerts,
        "Non-interruptive/audit queue": audit_queue,
        "Alert burden reduction (%)": burden_reduction,
        "Serious ADE sensitivity retained (%)": sensitivity_retained,
        "Active alert PPV (%)": ppv_active,
        "False positive active alerts": false_positive_active
    })

threshold_results_strict_df = pd.DataFrame(threshold_results_strict)

display(threshold_results_strict_df)

threshold_results_strict_df.to_csv("threshold_optimization_strict_leakage_reduced.csv", index=False)
print("Saved: threshold_optimization_strict_leakage_reduced.csv")

In [ ]:
def assign_alert_tier_safe(p):
    if p >= 0.80:
        return "Critical"
    elif p >= 0.50:
        return "High"
    elif p >= 0.30:
        return "Advisory"
    else:
        return "Low-risk audit queue"

alert_strict_df["alert_tier"] = alert_strict_df["predicted_risk"].apply(assign_alert_tier_safe)

alert_strict_summary = alert_strict_df.groupby("alert_tier").agg(
    total_alerts=("alert_tier", "count"),
    serious_ade_cases=("true_serious_ade", "sum"),
    mean_predicted_risk=("predicted_risk", "mean")
).reset_index()

alert_strict_summary["percent_of_alerts"] = (
    alert_strict_summary["total_alerts"] / len(alert_strict_df) * 100
)

alert_strict_summary["serious_ade_rate"] = (
    alert_strict_summary["serious_ade_cases"] /
    alert_strict_summary["total_alerts"] * 100
)

tier_order = ["Critical", "High", "Advisory", "Low-risk audit queue"]
alert_strict_summary["tier_order"] = alert_strict_summary["alert_tier"].map(
    {t: i for i, t in enumerate(tier_order)}
)

alert_strict_summary = (
    alert_strict_summary
    .sort_values("tier_order")
    .drop(columns=["tier_order"])
)

display(alert_strict_summary)

alert_strict_summary.to_csv("alert_tier_summary_strict_leakage_reduced.csv", index=False)
print("Saved: alert_tier_summary_strict_leakage_reduced.csv")

In [ ]:
import joblib

joblib.dump(xgb_model_strict, "xgboost_ade_model_strict_leakage_reduced.pkl")

print("Saved: xgboost_ade_model_strict_leakage_reduced.pkl")

In [ ]:
from pathlib import Path
import pandas as pd

important_strict_outputs = [
    "strict_leakage_sensitivity_analysis.csv",
    "threshold_optimization_strict_leakage_reduced.csv",
    "alert_tier_summary_strict_leakage_reduced.csv",
    "xgboost_ade_model_strict_leakage_reduced.pkl"
]

check_rows = []

for fname in important_strict_outputs:
    path = Path.cwd() / fname
    check_rows.append({
        "File": fname,
        "Exists": "✅ Yes" if path.exists() else "❌ No",
        "Size KB": round(path.stat().st_size / 1024, 2) if path.exists() else None
    })

strict_output_check = pd.DataFrame(check_rows)

display(strict_output_check)

strict_output_check.to_csv("strict_output_check.csv", index=False)
print("Saved: strict_output_check.csv")

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()

target_files = {
    "strict_leakage_comparison_df": "strict_leakage_sensitivity_analysis.csv",
    "threshold_results_strict_df": "threshold_optimization_strict_leakage_reduced.csv",
    "alert_strict_summary": "alert_tier_summary_strict_leakage_reduced.csv"
}

loaded_tables = {}

print("Current folder:", BASE_DIR)

for table_name, file_name in target_files.items():
    file_path = BASE_DIR / file_name
    
    print("\n" + "="*90)
    print(f"{table_name}  |  {file_name}")
    print("="*90)
    
    if file_path.exists():
        df_table = pd.read_csv(file_path)
        loaded_tables[table_name] = df_table
        
        print("Topildi:", file_path)
        print("Shape:", df_table.shape)
        display(df_table)
    else:
        print("Topilmadi:", file_name)
        print("Avval shu jadvalni yaratadigan strict leakage analysis cellni run qilish kerak.")

# Jupyter xotirasiga ham yuklaymiz
if "strict_leakage_comparison_df" in loaded_tables:
    strict_leakage_comparison_df = loaded_tables["strict_leakage_comparison_df"]

if "threshold_results_strict_df" in loaded_tables:
    threshold_results_strict_df = loaded_tables["threshold_results_strict_df"]

if "alert_strict_summary" in loaded_tables:
    alert_strict_summary = loaded_tables["alert_strict_summary"]

print("\nTayyor. Topilgan strict jadvallar Jupyter xotirasiga ham yuklandi.")

In [ ]:
# =========================
# 0. Libraries
# =========================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import shap
import joblib

print("Libraries loaded.")

In [ ]:
# =========================
# 1. Find original dataset
# =========================

BASE_DIR = Path.cwd()
print("Current folder:", BASE_DIR)

all_csv = list(BASE_DIR.glob("*.csv"))

exclude_keywords = [
    "model_results",
    "alert",
    "threshold",
    "fairness",
    "shap",
    "leakage",
    "temporal",
    "summary",
    "calibration",
    "output"
]

candidate_csv = [
    f for f in all_csv
    if not any(k in f.name.lower() for k in exclude_keywords)
]

print("\nOriginal database bo‘lishi mumkin bo‘lgan CSV fayllar:")
for i, f in enumerate(candidate_csv, start=1):
    print(i, f.name)

if len(candidate_csv) == 0:
    raise FileNotFoundError("Original dataset CSV topilmadi. Maqola papkasida database fayl borligini tekshiring.")

DATA_PATH = candidate_csv[0]
print("\nTanlangan dataset:", DATA_PATH.name)

In [ ]:
# =========================
# 2. Load dataset
# =========================

NROWS = 200000  # avval 200k bilan ishlaymiz; keyin full datasetga o'tish mumkin

df = pd.read_csv(DATA_PATH, nrows=NROWS, low_memory=False)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace(".", "_")
)

print("df shape:", df.shape)
print("\nColumns:")
for c in df.columns:
    print("-", c)

display(df.head())

In [ ]:
# =========================
# 3. Create/check target: serious_ade
# =========================

print("serious_ade mavjudmi?", "serious_ade" in df.columns)

if "serious_ade" in df.columns:
    df["serious_ade"] = pd.to_numeric(df["serious_ade"], errors="coerce").fillna(0).astype(int)
else:
    possible_label_cols = [
        c for c in df.columns
        if any(word in c for word in [
            "serious", "death", "hospital", "life_threat", "lifethreat",
            "disab", "congen", "fatal"
        ])
    ]

    print("Possible label columns:")
    for c in possible_label_cols:
        print("-", c)

    if len(possible_label_cols) == 0:
        raise ValueError("serious_ade yaratish uchun label ustunlari topilmadi.")

    for c in possible_label_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    df["serious_ade"] = (df[possible_label_cols].sum(axis=1) > 0).astype(int)

print("\nTarget distribution:")
print(df["serious_ade"].value_counts())
print(df["serious_ade"].value_counts(normalize=True))

In [ ]:
# =========================
# 4. Feature selection
# =========================

candidate_feature_cols = [
    "year",
    "num_reactions",
    "num_drugs",
    "patient_age_years",
    "patient_weight_kg",
    "report_age_days",
    "reactions",
    "primary_reaction",
    "reaction_outcomes",
    "suspect_drug",
    "drug_route",
    "drug_indication",
    "manufacturer",
    "drug_count_category",
    "age_group",
    "patient_sex",
    "country"
]

feature_cols = [c for c in candidate_feature_cols if c in df.columns]

print("Selected feature columns:")
for c in feature_cols:
    print("-", c)

model_df = df[feature_cols + ["serious_ade"]].copy()

print("\nmodel_df shape:", model_df.shape)
print("Target distribution:")
print(model_df["serious_ade"].value_counts(normalize=True))

display(model_df.head())

In [ ]:
# =========================
# 5. Primary model dataset: leakage-reduced
# remove reaction_outcomes
# =========================

leakage_features = ["reaction_outcomes"]

feature_cols_lr = [
    c for c in feature_cols
    if c not in leakage_features
]

X_lr = model_df[feature_cols_lr].copy()
y_lr = model_df["serious_ade"].astype(int).copy()

print("Original feature count:", len(feature_cols))
print("Leakage-reduced feature count:", len(feature_cols_lr))
print("Removed:", leakage_features)

print("X_lr shape:", X_lr.shape)
print("y_lr distribution:")
print(y_lr.value_counts(normalize=True))

In [ ]:
# =========================
# 6. Random held-out split
# =========================

X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr,
    y_lr,
    test_size=0.2,
    random_state=42,
    stratify=y_lr
)

print("Train:", X_lr_train.shape)
print("Test:", X_lr_test.shape)

print("\nTrain target:")
print(y_lr_train.value_counts(normalize=True))

print("\nTest target:")
print(y_lr_test.value_counts(normalize=True))

In [ ]:
# =========================
# 7. Preprocessing
# =========================

categorical_cols_lr = X_lr_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_lr = X_lr_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical columns:", categorical_cols_lr)
print("Numeric columns:", numeric_cols_lr)

numeric_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_cols_lr),
        ("cat", categorical_transformer_lr, categorical_cols_lr)
    ]
)

In [ ]:
# =========================
# 8. Train primary leakage-reduced XGBoost
# =========================

xgb_model_lr = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model_lr.fit(X_lr_train, y_lr_train)

xgb_lr_proba = xgb_model_lr.predict_proba(X_lr_test)[:, 1]
xgb_lr_pred = (xgb_lr_proba >= 0.5).astype(int)

print("Leakage-reduced XGBoost AUROC:", roc_auc_score(y_lr_test, xgb_lr_proba))
print("Leakage-reduced XGBoost AUPRC:", average_precision_score(y_lr_test, xgb_lr_proba))
print(classification_report(y_lr_test, xgb_lr_pred))
print(confusion_matrix(y_lr_test, xgb_lr_pred))

In [ ]:
# =========================
# 9. Metric helper
# =========================

def specificity_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score_from_cm(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def get_metrics(model_name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "Model": model_name,
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score_from_cm(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score_from_cm(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

primary_results_df = pd.DataFrame([
    get_metrics("Primary leakage-reduced XGBoost", y_lr_test, xgb_lr_proba)
])

display(primary_results_df)

primary_results_df.to_csv("primary_leakage_reduced_model_results.csv", index=False)
print("Saved: primary_leakage_reduced_model_results.csv")

In [ ]:
# =========================
# 10. Strict leakage-reduced dataset
# =========================

strict_leakage_features = [
    "reaction_outcomes",
    "primary_reaction",
    "reactions"
]

feature_cols_strict = [
    c for c in feature_cols
    if c not in strict_leakage_features
]

X_strict = model_df[feature_cols_strict].copy()
y_strict = model_df["serious_ade"].astype(int).copy()

print("Strict feature count:", len(feature_cols_strict))
print("Removed:", strict_leakage_features)
print("X_strict:", X_strict.shape)

print("\nStrict features:")
for c in feature_cols_strict:
    print("-", c)

In [ ]:
# =========================
# 11. Strict model train/test
# =========================

X_strict_train, X_strict_test, y_strict_train, y_strict_test = train_test_split(
    X_strict,
    y_strict,
    test_size=0.2,
    random_state=42,
    stratify=y_strict
)

categorical_cols_strict = X_strict_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_strict = X_strict_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

numeric_transformer_strict = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_strict = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_strict = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_strict, numeric_cols_strict),
        ("cat", categorical_transformer_strict, categorical_cols_strict)
    ]
)

xgb_model_strict = Pipeline(steps=[
    ("preprocessor", preprocessor_strict),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model_strict.fit(X_strict_train, y_strict_train)

xgb_strict_proba = xgb_model_strict.predict_proba(X_strict_test)[:, 1]
xgb_strict_pred = (xgb_strict_proba >= 0.5).astype(int)

print("Strict leakage-reduced XGBoost AUROC:", roc_auc_score(y_strict_test, xgb_strict_proba))
print("Strict leakage-reduced XGBoost AUPRC:", average_precision_score(y_strict_test, xgb_strict_proba))
print(classification_report(y_strict_test, xgb_strict_pred))
print(confusion_matrix(y_strict_test, xgb_strict_pred))

In [ ]:
# =========================
# 12. Leakage sensitivity comparison
# =========================

full_model_metrics = {
    "Model": "Full XGBoost with reaction_outcomes",
    "AUROC": 0.865324,
    "AUPRC": 0.841289,
    "Sensitivity": 0.738570,
    "Specificity": 0.808544,
    "PPV": 0.761037,
    "NPV": 0.789307,
    "F1": 0.749635,
    "Brier Score": 0.149629
}

primary_metrics = get_metrics(
    "Primary leakage-reduced XGBoost without reaction_outcomes",
    y_lr_test,
    xgb_lr_proba
)

strict_metrics = get_metrics(
    "Strict leakage-reduced XGBoost without reaction text fields",
    y_strict_test,
    xgb_strict_proba
)

strict_leakage_comparison_df = pd.DataFrame([
    full_model_metrics,
    primary_metrics,
    strict_metrics
])

display(strict_leakage_comparison_df)

strict_leakage_comparison_df.to_csv("strict_leakage_sensitivity_analysis.csv", index=False)
print("Saved: strict_leakage_sensitivity_analysis.csv")

In [ ]:
# =========================
# 13. Automatic temporal split
# =========================

temporal_df = model_df.copy()
temporal_df["year"] = pd.to_numeric(temporal_df["year"], errors="coerce")
temporal_df = temporal_df.dropna(subset=["year"]).copy()
temporal_df["year"] = temporal_df["year"].astype(int)

available_years = sorted(temporal_df["year"].unique())
print("Available years:", available_years)

if len(available_years) >= 3:
    test_year = available_years[-1]
    valid_year = available_years[-2]
    train_years = available_years[:-2]

    train_temporal = temporal_df[temporal_df["year"].isin(train_years)].copy()
    valid_temporal = temporal_df[temporal_df["year"] == valid_year].copy()
    test_temporal = temporal_df[temporal_df["year"] == test_year].copy()
else:
    temporal_df = temporal_df.sort_values("year").reset_index(drop=True)
    n = len(temporal_df)
    train_end = int(n * 0.60)
    valid_end = int(n * 0.80)

    train_temporal = temporal_df.iloc[:train_end].copy()
    valid_temporal = temporal_df.iloc[train_end:valid_end].copy()
    test_temporal = temporal_df.iloc[valid_end:].copy()

print("Train temporal:", train_temporal.shape)
print("Validation temporal:", valid_temporal.shape)
print("Test temporal:", test_temporal.shape)

print("\nTrain years:")
print(train_temporal["year"].value_counts().sort_index())

print("\nValidation years:")
print(valid_temporal["year"].value_counts().sort_index())

print("\nTest years:")
print(test_temporal["year"].value_counts().sort_index())

In [ ]:
# =========================
# 14. Temporal validation training
# Use primary leakage-reduced features
# =========================

if len(train_temporal) == 0 or len(valid_temporal) == 0 or len(test_temporal) == 0:
    raise ValueError("Temporal split bo‘sh chiqdi. Year ustunini tekshiring.")

temporal_feature_cols = feature_cols_lr.copy()

X_train_t = train_temporal[temporal_feature_cols].copy()
y_train_t = train_temporal["serious_ade"].astype(int).copy()

X_valid_t = valid_temporal[temporal_feature_cols].copy()
y_valid_t = valid_temporal["serious_ade"].astype(int).copy()

X_test_t = test_temporal[temporal_feature_cols].copy()
y_test_t = test_temporal["serious_ade"].astype(int).copy()

categorical_cols_t = X_train_t.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_cols_t = X_train_t.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

numeric_transformer_t = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_t = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50))
])

preprocessor_t = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_t, numeric_cols_t),
        ("cat", categorical_transformer_t, categorical_cols_t)
    ]
)

xgb_temporal_model = Pipeline(steps=[
    ("preprocessor", preprocessor_t),
    ("model", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_temporal_model.fit(X_train_t, y_train_t)

valid_t_proba = xgb_temporal_model.predict_proba(X_valid_t)[:, 1]
test_t_proba = xgb_temporal_model.predict_proba(X_test_t)[:, 1]

temporal_results_df = pd.DataFrame([
    get_metrics("Temporal validation set", y_valid_t, valid_t_proba),
    get_metrics("Temporal test set", y_test_t, test_t_proba)
])

display(temporal_results_df)

temporal_results_df.to_csv("temporal_validation_results.csv", index=False)
print("Saved: temporal_validation_results.csv")

In [ ]:
# =========================
# 15. Threshold optimization: primary leakage-reduced model
# =========================

alert_lr_df = X_lr_test.copy()
alert_lr_df["true_serious_ade"] = y_lr_test.values
alert_lr_df["predicted_risk"] = xgb_lr_proba

threshold_results_lr = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    active = alert_lr_df["predicted_risk"] >= threshold
    
    total_alerts = len(alert_lr_df)
    active_alerts = int(active.sum())
    audit_queue = total_alerts - active_alerts
    burden_reduction = audit_queue / total_alerts * 100
    
    serious_total = int(alert_lr_df["true_serious_ade"].sum())
    serious_captured = int(alert_lr_df.loc[
        active & (alert_lr_df["true_serious_ade"] == 1)
    ].shape[0])
    
    sensitivity_retained = serious_captured / serious_total * 100
    
    false_positive_active = int(alert_lr_df.loc[
        active & (alert_lr_df["true_serious_ade"] == 0)
    ].shape[0])
    
    ppv_active = serious_captured / active_alerts * 100 if active_alerts > 0 else np.nan
    
    threshold_results_lr.append({
        "Model": "Primary leakage-reduced XGBoost",
        "Active alert threshold": threshold,
        "Total potential alerts": total_alerts,
        "Active alerts shown": active_alerts,
        "Non-interruptive/audit queue": audit_queue,
        "Alert burden reduction (%)": burden_reduction,
        "Serious ADE sensitivity retained (%)": sensitivity_retained,
        "Active alert PPV (%)": ppv_active,
        "False positive active alerts": false_positive_active
    })

threshold_results_lr_df = pd.DataFrame(threshold_results_lr)

display(threshold_results_lr_df)

threshold_results_lr_df.to_csv("threshold_optimization_leakage_reduced.csv", index=False)
print("Saved: threshold_optimization_leakage_reduced.csv")

In [ ]:
# =========================
# 16. Alert tier summary
# =========================

def assign_alert_tier_safe(p):
    if p >= 0.80:
        return "Critical"
    elif p >= 0.50:
        return "High"
    elif p >= 0.30:
        return "Advisory"
    else:
        return "Low-risk audit queue"

alert_lr_df["alert_tier"] = alert_lr_df["predicted_risk"].apply(assign_alert_tier_safe)

alert_lr_summary = alert_lr_df.groupby("alert_tier").agg(
    total_alerts=("alert_tier", "count"),
    serious_ade_cases=("true_serious_ade", "sum"),
    mean_predicted_risk=("predicted_risk", "mean")
).reset_index()

alert_lr_summary["percent_of_alerts"] = (
    alert_lr_summary["total_alerts"] / len(alert_lr_df) * 100
)

alert_lr_summary["serious_ade_rate"] = (
    alert_lr_summary["serious_ade_cases"] /
    alert_lr_summary["total_alerts"] * 100
)

tier_order = ["Critical", "High", "Advisory", "Low-risk audit queue"]

alert_lr_summary["tier_order"] = alert_lr_summary["alert_tier"].map(
    {t: i for i, t in enumerate(tier_order)}
)

alert_lr_summary = (
    alert_lr_summary
    .sort_values("tier_order")
    .drop(columns=["tier_order"])
)

display(alert_lr_summary)

alert_lr_summary.to_csv("alert_tier_summary_leakage_reduced.csv", index=False)
print("Saved: alert_tier_summary_leakage_reduced.csv")

In [ ]:
# =========================
# 17. Fairness / subgroup analysis
# =========================

fairness_lr_df = X_lr_test.copy()
fairness_lr_df["true_serious_ade"] = y_lr_test.values
fairness_lr_df["predicted_risk"] = xgb_lr_proba
fairness_lr_df["predicted_label"] = (xgb_lr_proba >= 0.5).astype(int)

def subgroup_metrics_safe(data, subgroup_col, min_group_size=100):
    rows = []

    if subgroup_col not in data.columns:
        print(f"Column not found: {subgroup_col}")
        return pd.DataFrame()

    for group, g in data.groupby(subgroup_col, dropna=False):
        if len(g) < min_group_size:
            continue

        y_true_g = g["true_serious_ade"]
        y_prob_g = g["predicted_risk"]
        y_pred_g = g["predicted_label"]

        if y_true_g.nunique() < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(y_true_g, y_prob_g)
            auprc = average_precision_score(y_true_g, y_prob_g)

        tn, fp, fn, tp = confusion_matrix(
            y_true_g,
            y_pred_g,
            labels=[0, 1]
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
        f1 = f1_score(y_true_g, y_pred_g, zero_division=0)
        brier = brier_score_loss(y_true_g, y_prob_g)

        rows.append({
            "Subgroup variable": subgroup_col,
            "Subgroup": group,
            "n": len(g),
            "ADE rate (%)": y_true_g.mean() * 100,
            "AUROC": auroc,
            "AUPRC": auprc,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "PPV": ppv,
            "NPV": npv,
            "F1": f1,
            "FNR": fnr,
            "FPR": fpr,
            "Brier Score": brier
        })

    result = pd.DataFrame(rows)

    if len(result) > 0:
        result["AUROC gap vs best"] = result["AUROC"].max() - result["AUROC"]
        result["Sensitivity gap vs best"] = result["Sensitivity"].max() - result["Sensitivity"]
        result["FNR gap vs lowest"] = result["FNR"] - result["FNR"].min()

        result["Fairness flag"] = "No major concern"
        result.loc[result["AUROC gap vs best"] > 0.05, "Fairness flag"] = "AUROC gap > 0.05"
        result.loc[result["Sensitivity gap vs best"] > 0.10, "Fairness flag"] = "Sensitivity gap > 0.10"
        result.loc[result["FNR gap vs lowest"] > 0.10, "Fairness flag"] = "FNR gap > 0.10"

    return result

fairness_lr_tables = []

for col in ["patient_sex", "age_group", "country", "drug_count_category"]:
    if col in fairness_lr_df.columns:
        if col == "country":
            top_countries = fairness_lr_df["country"].value_counts().head(10).index
            temp_df = fairness_lr_df[fairness_lr_df["country"].isin(top_countries)].copy()
            result = subgroup_metrics_safe(temp_df, col, min_group_size=100)
        else:
            result = subgroup_metrics_safe(fairness_lr_df, col, min_group_size=100)

        fairness_lr_tables.append(result)
        print(f"\n=== Fairness by {col} ===")
        display(result)

fairness_lr_all = pd.concat(
    [t for t in fairness_lr_tables if len(t) > 0],
    ignore_index=True
)

fairness_lr_summary = fairness_lr_all[[
    "Subgroup variable",
    "Subgroup",
    "n",
    "ADE rate (%)",
    "AUROC",
    "AUPRC",
    "Sensitivity",
    "Specificity",
    "FNR",
    "FPR",
    "Brier Score",
    "Fairness flag"
]].copy()

display(fairness_lr_summary)

fairness_lr_summary.to_csv("fairness_summary_leakage_reduced.csv", index=False)
print("Saved: fairness_summary_leakage_reduced.csv")

In [ ]:
# =========================
# 18. SHAP global explanation
# =========================

xgb_lr_preprocessor = xgb_model_lr.named_steps["preprocessor"]
xgb_lr_estimator = xgb_model_lr.named_steps["model"]

shap_sample_size = 2000

X_lr_shap = X_lr_test.sample(
    n=min(shap_sample_size, len(X_lr_test)),
    random_state=42
)

X_lr_shap_transformed = xgb_lr_preprocessor.transform(X_lr_shap)

numeric_feature_names_lr = numeric_cols_lr

cat_encoder_lr = xgb_lr_preprocessor.named_transformers_["cat"].named_steps["onehot"]
categorical_feature_names_lr = cat_encoder_lr.get_feature_names_out(categorical_cols_lr).tolist()

feature_names_lr = numeric_feature_names_lr + categorical_feature_names_lr

print("Feature names count:", len(feature_names_lr))
print("Transformed columns:", X_lr_shap_transformed.shape[1])

explainer_lr = shap.TreeExplainer(xgb_lr_estimator)
shap_values_lr = explainer_lr.shap_values(X_lr_shap_transformed)

print("SHAP values shape:", np.array(shap_values_lr).shape)

In [ ]:
# =========================
# 19. SHAP plots and feature table
# =========================

plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_lr_shap_transformed,
    feature_names=feature_names_lr,
    show=False,
    max_display=20
)
plt.tight_layout()
plt.savefig("shap_summary_plot_leakage_reduced.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_lr_shap_transformed,
    feature_names=feature_names_lr,
    plot_type="bar",
    show=False,
    max_display=20
)
plt.tight_layout()
plt.savefig("shap_feature_importance_bar_leakage_reduced.png", dpi=300, bbox_inches="tight")
plt.show()

mean_abs_shap_lr = np.abs(shap_values_lr).mean(axis=0)

shap_importance_lr_df = pd.DataFrame({
    "Feature": feature_names_lr,
    "Mean_abs_SHAP": mean_abs_shap_lr
}).sort_values("Mean_abs_SHAP", ascending=False)

top20_shap_lr = shap_importance_lr_df.head(20).copy()
top20_shap_lr["Rank"] = range(1, len(top20_shap_lr) + 1)
top20_shap_lr = top20_shap_lr[["Rank", "Feature", "Mean_abs_SHAP"]]

display(top20_shap_lr)

shap_importance_lr_df.to_csv("shap_feature_importance_leakage_reduced.csv", index=False)
top20_shap_lr.to_csv("top20_shap_features_leakage_reduced.csv", index=False)

print("Saved SHAP outputs.")

In [ ]:
# =========================
# 20. Calibration curve
# =========================

prob_true, prob_pred = calibration_curve(
    y_lr_test,
    xgb_lr_proba,
    n_bins=10,
    strategy="quantile"
)

calibration_df = pd.DataFrame({
    "mean_predicted_probability": prob_pred,
    "observed_serious_ADE_rate": prob_true
})

display(calibration_df)

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker="o", label="Primary leakage-reduced XGBoost")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed serious ADE rate")
plt.title("Calibration curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("calibration_curve_leakage_reduced.png", dpi=300)
plt.show()

calibration_df.to_csv("calibration_curve_leakage_reduced.csv", index=False)

print("Saved: calibration_curve_leakage_reduced.png")
print("Saved: calibration_curve_leakage_reduced.csv")

In [ ]:
# =========================
# 21. Save models
# =========================

joblib.dump(xgb_model_lr, "xgboost_ade_model_leakage_reduced.pkl")
joblib.dump(xgb_model_strict, "xgboost_ade_model_strict_leakage_reduced.pkl")
joblib.dump(xgb_temporal_model, "xgboost_ade_model_temporal.pkl")

print("Saved:")
print("- xgboost_ade_model_leakage_reduced.pkl")
print("- xgboost_ade_model_strict_leakage_reduced.pkl")
print("- xgboost_ade_model_temporal.pkl")

In [ ]:
# =========================
# 22. Final output check
# =========================

important_outputs = [
    "primary_leakage_reduced_model_results.csv",
    "strict_leakage_sensitivity_analysis.csv",
    "temporal_validation_results.csv",
    "threshold_optimization_leakage_reduced.csv",
    "alert_tier_summary_leakage_reduced.csv",
    "fairness_summary_leakage_reduced.csv",
    "shap_summary_plot_leakage_reduced.png",
    "shap_feature_importance_bar_leakage_reduced.png",
    "top20_shap_features_leakage_reduced.csv",
    "calibration_curve_leakage_reduced.png",
    "calibration_curve_leakage_reduced.csv",
    "xgboost_ade_model_leakage_reduced.pkl",
    "xgboost_ade_model_strict_leakage_reduced.pkl",
    "xgboost_ade_model_temporal.pkl"
]

check_rows = []

for fname in important_outputs:
    path = Path.cwd() / fname
    check_rows.append({
        "File": fname,
        "Exists": "✅ Yes" if path.exists() else "❌ No",
        "Size KB": round(path.stat().st_size / 1024, 2) if path.exists() else None
    })

final_output_check = pd.DataFrame(check_rows)

display(final_output_check)

final_output_check.to_csv("final_output_check.csv", index=False)

print("Saved: final_output_check.csv")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__n_estimators": [300, 400, 600, 800],
    "model__max_depth": [3, 4, 5, 6, 7],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__gamma": [0, 0.1, 0.3, 0.5],
    "model__reg_alpha": [0, 0.01, 0.1, 1],
    "model__reg_lambda": [1, 2, 5, 10]
}

search_model = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

random_search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",  # AUPRC optimize qiladi
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_lr_train, y_lr_train)

print("Best params:")
print(random_search.best_params_)

best_xgb_model = random_search.best_estimator_

best_proba = best_xgb_model.predict_proba(X_lr_test)[:, 1]

print("Best tuned XGBoost AUROC:", roc_auc_score(y_lr_test, best_proba))
print("Best tuned XGBoost AUPRC:", average_precision_score(y_lr_test, best_proba))

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# Preprocessordan keyin XGBoostni alohida calibration qilish murakkabroq,
# shuning uchun avval pipeline model tayyor bo‘lsin.
calibrated_model = CalibratedClassifierCV(
    estimator=xgb_model_lr,
    method="isotonic",
    cv=3
)

calibrated_model.fit(X_lr_train, y_lr_train)

calibrated_proba = calibrated_model.predict_proba(X_lr_test)[:, 1]

print("Calibrated AUROC:", roc_auc_score(y_lr_test, calibrated_proba))
print("Calibrated AUPRC:", average_precision_score(y_lr_test, calibrated_proba))
print("Calibrated Brier Score:", brier_score_loss(y_lr_test, calibrated_proba))

In [17]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

def bootstrap_ci(
    y_true,
    y_prob,
    threshold=0.5,
    n_bootstrap=1000,
    ci=95,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    n = len(y_true)

    point_metrics = compute_metrics(y_true, y_prob, threshold=threshold)

    bootstrap_results = {k: [] for k in point_metrics.keys()}

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)

        y_true_b = y_true[idx]
        y_prob_b = y_prob[idx]

        # Bootstrap sample ichida faqat bitta class qolib ketsa, AUROC chiqmaydi
        if len(np.unique(y_true_b)) < 2:
            continue

        try:
            metrics_b = compute_metrics(y_true_b, y_prob_b, threshold=threshold)

            for k, v in metrics_b.items():
                bootstrap_results[k].append(v)

        except Exception:
            continue

    alpha = (100 - ci) / 2

    rows = []

    for metric_name, point_value in point_metrics.items():
        values = np.array(bootstrap_results[metric_name], dtype=float)
        values = values[~np.isnan(values)]

        lower = np.percentile(values, alpha)
        upper = np.percentile(values, 100 - alpha)

        rows.append({
            "Metric": metric_name,
            "Point estimate": point_value,
            f"{ci}% CI lower": lower,
            f"{ci}% CI upper": upper,
            "Formatted": f"{point_value:.3f} ({lower:.3f}–{upper:.3f})"
        })

    return pd.DataFrame(rows)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix
)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity_score(y_true, y_pred),
        "PPV": precision_score(y_true, y_pred, zero_division=0),
        "NPV": npv_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

def bootstrap_ci(
    y_true,
    y_prob,
    threshold=0.5,
    n_bootstrap=1000,
    ci=95,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    n = len(y_true)

    point_metrics = compute_metrics(y_true, y_prob, threshold=threshold)

    bootstrap_results = {k: [] for k in point_metrics.keys()}

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)

        y_true_b = y_true[idx]
        y_prob_b = y_prob[idx]

        # Bootstrap sample ichida faqat bitta class qolib ketsa, AUROC chiqmaydi
        if len(np.unique(y_true_b)) < 2:
            continue

        try:
            metrics_b = compute_metrics(y_true_b, y_prob_b, threshold=threshold)

            for k, v in metrics_b.items():
                bootstrap_results[k].append(v)

        except Exception:
            continue

    alpha = (100 - ci) / 2

    rows = []

    for metric_name, point_value in point_metrics.items():
        values = np.array(bootstrap_results[metric_name], dtype=float)
        values = values[~np.isnan(values)]

        lower = np.percentile(values, alpha)
        upper = np.percentile(values, 100 - alpha)

        rows.append({
            "Metric": metric_name,
            "Point estimate": point_value,
            f"{ci}% CI lower": lower,
            f"{ci}% CI upper": upper,
            "Formatted": f"{point_value:.3f} ({lower:.3f}–{upper:.3f})"
        })

    return pd.DataFrame(rows)